# 中国六大交易所 期货/期权 全历史行情：下载 → 解析 → 汇总

一个 notebook 走完三步：**下载原始年度包 → 解析成统一 schema → 按年合并落 parquet**，
最后附主力/次主力合约标注。

## 数据源（六个，不是五个）

| 来源 | 通道 | 接口 | 实测覆盖 |
|---|---|---|---|
| 上期所 SHFE | `requests` | `download.json` → 年度 zip；`kx{日期}.dat` 补当月 | 期货 2010–，期权 2018-09-21– |
| **能源中心 INE** | `requests` | 自己的 `download.json`（`www.ine.cn`） | 2018-03-26–，期权 2021-06-21– |
| 郑商所 CZCE | 年度包走 **Chrome**，日 txt 直连 | 三段式 URL，见第 5 节 | 期货 2010–，期权 2017-04-19– |
| 大商所 DCE | 全部走 **Chrome** | `/dcereport/quote/history/download` | 2010–，期权 2017-03-31– |
| 中金所 CFFEX | `requests` | 月度 zip | 2010-04-16– |
| 广期所 GFEX | `requests` | 年度 csv | 2022-12-22– |

**INE 必须单独抓。** 上期所的年度包在 2018–2023 完全不含能源中心，
少了这一步 `sc / lu / nr / bc` 会整整空六年。（2024 起上期所的包里才带上 ine 目录，
所以本 notebook 里 SHFE 解析**永远跳过**包内 ine 目录，INE 一律从独立包读，避免重复计数。）

## 为什么 CZCE / DCE 要走真实 Chrome

这两家在 Akamai Bot Manager 后面，拦截时返回 `412` + 一个只藏 sensor 链接的空 HTML。
同一台机器同一条网络上实测：`requests` 412、`curl_cffi` 伪装 TLS 指纹**照样 412**、
自动化 Chromium 拿不到 cookie，**只有本机真实 Chrome 全部 200**。
第 2 节的 `ChromeDL` 就是干这个的，零额外依赖。

## 重复下载？不会，但当年的包必须刷新

- 所有下载都是**先看本地文件在不在、够不够大，在就跳过**（打印 `skip`），中断重跑不会重复下载。
- 但**当年的包交易所还在往里追加**，跳过就等于永远停在旧版本。
  所以 `_ow()` 会对 `YEAR_TO`（当年）强制覆盖重下，往年已定稿的才跳过。
- 还有一层滞后：SHFE/INE 的年度包更新比行情慢（当前包名就叫「2026.1月-7月」），
  最近一两个月不在包里。第 12 节用日更 `.dat` 把这个尾巴补上。

## 依赖

```bash
python -m pip install xlrd "openpyxl>=3.1.5" python-calamine pyarrow
```

`python-calamine` 不是可选的：DCE 那个 95 MB 的 xlsx 用 openpyxl 要 62 秒，calamine 只要 4.2 秒。

## 目录

```
<project-root>\
  raw\      shfe\ ine\ czce\ dce\ cffex\ gfex\        原始包，约 1.33 GB
  parsed\   futures\futures_{年}.parquet              约  62 MB
            options\options_{年}.parquet              约 390 MB
```

## 1. 公共配置与工具函数

In [ ]:
import os, re, io, json, time, shutil, zipfile, subprocess, urllib.request
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------- 全局配置 ----------
BASE      = Path(os.environ.get("FUTURES_REPORT_BASE", str(Path.cwd()))).resolve()
ROOT      = BASE / "raw"            # 原始包
PARSED    = BASE / "parsed"         # 解析产物
YEAR_FROM = 2010
YEAR_TO   = dt.date.today().year
TIMEOUT   = 180
PAUSE     = 0.8

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36")


def make_session(referer=None, retries=4):
    """带重试的 Session。只对 5xx/429 重试；404/412 立即返回（它们不是偶发错误）。"""
    s = requests.Session()
    s.headers.update({"User-Agent": UA, "Accept": "*/*",
                      "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
                      "Connection": "keep-alive"})
    if referer:
        s.headers["Referer"] = referer
    retry = Retry(total=retries, backoff_factor=1.5,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=frozenset(["GET", "POST"]))
    ad = HTTPAdapter(max_retries=retry, pool_maxsize=8)
    s.mount("http://", ad); s.mount("https://", ad)
    return s


def _ow(overwrite, year):
    """当年的包交易所还在往里追加，必须覆盖重下；往年的已定稿，可以跳过。

    没有这一层的话，重跑 download_all() 会因为"本地已有文件"而永远停在
    当年那份的旧快照上 —— 文件在、大小也够，但内容是过时的。
    """
    return overwrite or int(year) >= YEAR_TO


def fetch_to(sess, url, path, min_size=512, overwrite=False, expect=None, **kw):
    """requests 下载。overwrite=False 且本地文件够大 -> 直接跳过（断点续传靠这个）。"""
    path = Path(path)
    if (not overwrite) and path.exists() and path.stat().st_size >= min_size:
        return "skip"
    try:
        request_timeout = kw.pop("timeout", TIMEOUT)
        r = sess.get(url, timeout=request_timeout, **kw)
    except Exception as e:
        return "ERR %s: %s" % (type(e).__name__, e)

    if r.status_code == 404:
        return "404"                          # 非交易日 / 该年无数据，属正常
    if r.status_code == 412:
        return "BLOCKED(412)"                 # Akamai，得改走 ChromeDL
    if r.status_code != 200:
        return "HTTP %s" % r.status_code
    if len(r.content) < min_size:
        return "TOO_SMALL %d" % len(r.content)
    if expect == "zip" and r.content[:2] != b"PK":
        return "NOT_ZIP"
    if expect == "json":
        try:
            json.loads(r.content.decode("utf-8"))
        except Exception:
            return "NOT_JSON"

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(r.content)
    time.sleep(PAUSE)
    return "ok %.2fMB" % (len(r.content) / 1e6)


BAD_PREFIXES = ("ERR", "BLOCKED", "HTTP", "NOT_", "TOO_SMALL", "TIMEOUT", "NO_FILE", "CANCELED")


def log(tag, name, result):
    bad = result.startswith(BAD_PREFIXES)
    print("%s[%s] %-34s %s" % ("!! " if bad else "   ", tag, name, result))
    return not bad


ROOT.mkdir(parents=True, exist_ok=True)
print("原始目录:", ROOT, "\n解析目录:", PARSED, "\n年份范围:", YEAR_FROM, "-", YEAR_TO)

## 2. `ChromeDL` —— 驱动本机真实 Chrome（CZCE / DCE 专用）

1. 用**独立临时 profile** 启动本机 Chrome（不碰你日常 profile 和登录状态）
2. `Browser.setDownloadBehavior` 把下载目录指到临时文件夹并打开下载事件
3. `warm(url)` 打开交易所页面，等 Akamai sensor 跑完
4. `get(url, dst)` 插一个隐藏 `<a download>` 点击 —— 走 Chrome 自己的下载栈，
   带完整 cookie 和指纹，**95 MB 的文件也不占 Python 内存**
5. 听 `Browser.downloadProgress` 等 `completed`，再把文件挪到目标路径

`get()` 同样有 skip 逻辑：本地文件在且够大就直接返回 `skip`，不会重复下。

**默认 `headless=False`，会弹一个 Chrome 窗口，别关它**，跑完自动退。

In [ ]:
import websocket   # websocket-client，anaconda base 自带

CHROME_CANDIDATES = [
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
    r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
] + [p for p in (shutil.which("google-chrome"), shutil.which("google-chrome-stable"),
                   shutil.which("chromium"), shutil.which("chromium-browser")) if p]


class ChromeDL:
    def __init__(self, workdir, port=9222, headless=None, chrome=None):
        # headless 留空时看环境变量：计划任务在没有交互桌面的会话里跑，必须 headless
        if headless is None:
            headless = os.environ.get("CHROMEDL_HEADLESS", "") == "1"
        self.work = Path(workdir); self.work.mkdir(parents=True, exist_ok=True)
        self.prof = self.work / "profile"
        self.dl   = self.work / "downloads"; self.dl.mkdir(parents=True, exist_ok=True)
        self.port, self.headless = port, headless
        self.chrome = chrome or next((c for c in CHROME_CANDIDATES if os.path.exists(c)), None)
        if not self.chrome:
            raise RuntimeError("找不到 Chrome/Edge，请手动传 chrome=r'...chrome.exe'")
        self.proc = self.ws = None
        self._id = 0

    def start(self):
        args = [self.chrome, "--remote-debugging-port=%d" % self.port,
                "--user-data-dir=%s" % self.prof, "--no-first-run",
                "--no-default-browser-check", "--disable-popup-blocking",
                "--disable-dev-shm-usage", "--window-size=1200,860", "about:blank"]
        if self.headless:
            # **--user-agent 不能省**：headless 默认 UA 带 "HeadlessChrome"，
            # Akamai 认这个字样，郑商所/大商所会一直拿不到文件（实测 0 字节）。
            # 补上真实 UA 之后，两家拿到的字节数跟有窗口模式完全一致。
            args.insert(1, "--headless=new")
            args.insert(2, "--user-agent=" + UA)
        self.proc = subprocess.Popen(args, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(80):
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/json/version" % self.port,
                                       timeout=1).read()
                break
            except Exception:
                time.sleep(0.5)
        else:
            raise RuntimeError("Chrome 起不来")

        req = urllib.request.Request(          # 新版 Chrome 的 /json/new 只收 PUT
            "http://127.0.0.1:%d/json/new?about:blank" % self.port, data=b"", method="PUT")
        tgt = json.loads(urllib.request.urlopen(req, timeout=10).read())
        # suppress_origin 必须加：Chrome 会 403 掉带 Origin 头的 DevTools WS 握手
        self.ws = websocket.create_connection(tgt["webSocketDebuggerUrl"], timeout=300,
                                              suppress_origin=True, enable_multithread=True)
        self._send("Page.enable"); self._send("Runtime.enable")
        self._send("Browser.setDownloadBehavior", behavior="allow",
                   downloadPath=str(self.dl), eventsEnabled=True)
        return self

    def stop(self, wipe_profile=True):
        # 必须让整棵进程树退干净：渲染子进程会一直占着 profile 目录，
        # 只 terminate() 父进程的话 profile 删不掉，下次启动撞 lockfile。
        try: self._send("Browser.close")
        except Exception: pass
        try: self.ws and self.ws.close()
        except Exception: pass
        if self.proc:
            try:
                self.proc.wait(timeout=10)
            except Exception:
                if os.name == "nt":
                    subprocess.run(["taskkill", "/F", "/T", "/PID", str(self.proc.pid)],
                                   capture_output=True)
                else:
                    self.proc.kill()
            time.sleep(1)
        if wipe_profile:
            shutil.rmtree(self.prof, ignore_errors=True)

    def __enter__(self):  return self.start()
    def __exit__(self, *a): self.stop()

    def _send(self, method, **params):
        self._id += 1; mid = self._id
        self.ws.send(json.dumps({"id": mid, "method": method, "params": params}))
        while True:
            msg = json.loads(self.ws.recv())
            if msg.get("id") == mid:
                if "error" in msg:
                    raise RuntimeError("%s: %s" % (method, msg["error"]))
                return msg.get("result", {})

    def _js(self, expr, timeout=180):
        r = self._send("Runtime.evaluate", expression=expr, awaitPromise=True,
                       returnByValue=True, timeout=timeout * 1000)
        if r.get("exceptionDetails"):
            raise RuntimeError("JS: " + json.dumps(r["exceptionDetails"], ensure_ascii=False)[:300])
        return r["result"].get("value")

    def warm(self, url, wait=40):
        """打开页面，等 Akamai 挑战跑完（正文渲染出来 = 过了）。"""
        self._send("Page.navigate", url=url)
        for _ in range(wait):
            time.sleep(1)
            try:
                self._send("Runtime.enable")
                n = self._js("document.body?document.body.innerText.length:0")
            except Exception:
                continue            # 挑战页会自跳转，跳转瞬间 evaluate 必然失败
            if isinstance(n, int) and n > 300:
                return True
        return False

    def get(self, url, dst, timeout=600, min_size=1024, overwrite=False):
        dst = Path(dst)
        if (not overwrite) and dst.exists() and dst.stat().st_size >= min_size:
            return "skip"

        before = {p.name for p in self.dl.iterdir()}
        self._js("(()=>{const a=document.createElement('a');a.href=%s;"
                 "a.download='';a.style.display='none';document.body.appendChild(a);"
                 "a.click();})()" % json.dumps(url))

        guid = name = None
        deadline = time.time() + timeout
        self.ws.settimeout(5)
        try:
            while time.time() < deadline:
                try:
                    msg = json.loads(self.ws.recv())
                except Exception:
                    continue        # recv 超时，继续等下载事件
                m, p = msg.get("method"), msg.get("params", {})
                if m == "Browser.downloadWillBegin":
                    guid, name = p.get("guid"), p.get("suggestedFilename")
                elif m == "Browser.downloadProgress" and p.get("guid") == guid:
                    if p.get("state") == "completed": break
                    if p.get("state") == "canceled":
                        return "CANCELED（多半被 412 挡了 / 该年无此文件）"
            else:
                return "TIMEOUT"
        finally:
            self.ws.settimeout(300)

        cands = [self.dl / name] if name and (self.dl / name).exists() else \
                [self.dl / n for n in ({p.name for p in self.dl.iterdir()} - before)]
        cands = [c for c in cands if c.is_file() and not c.name.endswith(".crdownload")]
        if not cands:
            return "NO_FILE"
        src = max(cands, key=lambda p: p.stat().st_mtime)
        if src.stat().st_size < min_size:
            src.unlink(missing_ok=True); return "TOO_SMALL"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        return "ok %.2fMB" % (dst.stat().st_size / 1e6)

    def post_json(self, url, payload, dst, timeout=120, min_size=200, overwrite=False):
        """在已通过挑战的页面上下文 POST；Cookie/Origin 由 Chrome 自动携带。"""
        dst = Path(dst)
        if (not overwrite) and dst.exists() and dst.stat().st_size >= min_size:
            return "skip"
        expr = ("(async()=>{const r=await fetch(%s,{method:'POST',"
                "headers:{'Accept':'application/json, text/plain, */*',"
                "'Content-Type':'application/json'},credentials:'same-origin',"
                "body:JSON.stringify(%s)});const text=await r.text();"
                "return {status:r.status,text};})()"
                % (json.dumps(url), json.dumps(payload, ensure_ascii=False)))
        out = self._js(expr, timeout=timeout)
        if not isinstance(out, dict) or out.get("status") != 200:
            return "HTTP %s" % (out.get("status") if isinstance(out, dict) else "?")
        text = out.get("text") or ""
        try:
            doc = json.loads(text)
        except Exception:
            return "NOT_JSON"
        if (doc.get("success") is not True or doc.get("code") != 200
                or not isinstance(doc.get("data"), list)):
            return "TOO_SMALL_OR_BAD_JSON"
        if not doc["data"]:
            return "EMPTY"             # 合法响应；是否休市由六所目标日数据共同判断
        if len(text.encode("utf-8")) < min_size:
            return "TOO_SMALL_OR_BAD_JSON"
        dst.parent.mkdir(parents=True, exist_ok=True)
        part = dst.with_suffix(dst.suffix + ".part")
        part.write_text(text, encoding="utf-8")
        os.replace(part, dst)
        return "ok %.2fMB" % (dst.stat().st_size / 1e6)


CHROME_WORK = BASE / ".chromedl"

## 3. 上期所 SHFE

年度包清单来自下载页同目录的 `download.json`：`type=1` 期货、`type=2` 期权。

**注意**：这两个 type 的 URL 不同，但服务端返回的是**同一个 zip**（md5 一致）。
那个包叫「所内合约行情报表」，期货和期权本来就在同一张表里，靠合约代码区分。
所以下面只下 type=1，不重复存一份。

In [ ]:
SHFE_HOST = "https://www.shfe.com.cn"
SHFE_DIR  = ROOT / "shfe"


def shfe_catalog():
    s = make_session(referer=SHFE_HOST + "/reports/tradedata/datadownload/")
    j = s.get(SHFE_HOST + "/reports/tradedata/datadownload/download.json", timeout=60).json()["data"]
    out, seen = [], set()
    for it in j.get("latestData", []) + j.get("historicalData", []):
        kind = "futures" if str(it["type"]) == "1" else "option"
        k = (int(it["year"]), kind)
        if k in seen:
            continue
        seen.add(k)
        out.append({"year": int(it["year"]), "kind": kind, "url": SHFE_HOST + it["url"]})
    return sorted(out, key=lambda x: (x["kind"], x["year"]))


def shfe_download_history(year_from=YEAR_FROM, year_to=YEAR_TO, overwrite=False):
    """只下 type=1 的包 —— type=2 是同一个文件，下了纯属浪费 284MB。"""
    s = make_session(referer=SHFE_HOST + "/reports/tradedata/datadownload/")
    ok = 0
    for it in shfe_catalog():
        if it["kind"] != "futures" or not (year_from <= it["year"] <= year_to):
            continue
        dst = SHFE_DIR / "history" / "futures" / ("SHFE_futures_%d.zip" % it["year"])
        res = fetch_to(s, it["url"], dst, min_size=10000, expect="zip",
                       overwrite=_ow(overwrite, it["year"]))
        ok += log("SHFE", "期货+期权 %d" % it["year"], res)
    return ok


def shfe_download_day(day, overwrite=True, options=True, futures=True):
    """日更 .dat（JSON）。始终刷新，避免盘中半截 raw 被永久 skip。"""
    s = make_session(referer=SHFE_HOST + "/reports/tradedata/dailyandweeklydata/", retries=1)
    ds = day.strftime("%Y%m%d")
    kinds = []
    if futures:
        kinds.append(("futures", "future"))
    if options:
        kinds.append(("option", "option"))
    results = {}
    for kind, seg in kinds:
        dst = SHFE_DIR / "daily" / kind / ("kx%s.dat" % ds)
        url = "%s/data/tradedata/%s/dailydata/kx%s.dat" % (SHFE_HOST, seg, ds)
        res = fetch_to(s, url, dst, min_size=200, expect="json", overwrite=overwrite,
                       timeout=(8, 45))
        log("SHFE", "%s %s" % (kind, ds), res)
        results[kind] = res
    return results

## 4. 能源中心 INE（`www.ine.cn`）

**这一节不能省。** 上期所的年度包 2018–2023 完全不含能源中心，
`sc` 原油 / `lu` 低硫燃料油 / `nr` 20号胶 / `bc` 国际铜 / `ec` 欧线集运指数 会整整空六年。

站点结构和上期所一模一样，也是 `download.json` + 年度 zip，期货/期权同样是同一个文件。
可用年份 2018–今（期权 2021 起，但因为同包所以不用单独下）。

In [ ]:
INE_HOST = "https://www.ine.cn"
INE_DIR  = ROOT / "ine"


def ine_catalog():
    s = make_session(referer=INE_HOST + "/reports/tradedata/datadownload/")
    j = s.get(INE_HOST + "/reports/tradedata/datadownload/download.json", timeout=60).json()["data"]
    out, seen = [], set()
    for it in j.get("latestData", []) + j.get("historicalData", []):
        kind = "futures" if str(it["type"]) == "1" else "option"
        k = (int(it["year"]), kind)
        if k in seen:
            continue
        seen.add(k)
        out.append({"year": int(it["year"]), "kind": kind, "url": INE_HOST + it["url"]})
    return sorted(out, key=lambda x: (x["kind"], x["year"]))


def ine_download_history(year_from=YEAR_FROM, year_to=YEAR_TO, overwrite=False):
    """同样只下 type=1；type=2 是同一个文件。"""
    s = make_session(referer=INE_HOST + "/reports/tradedata/datadownload/")
    ok = 0
    for it in ine_catalog():
        if it["kind"] != "futures" or not (year_from <= it["year"] <= year_to):
            continue
        dst = INE_DIR / "history" / ("INE_futures_%d.zip" % it["year"])
        res = fetch_to(s, it["url"], dst, min_size=10000, expect="zip",
                       overwrite=_ow(overwrite, it["year"]))
        ok += log("INE", "期货+期权 %d" % it["year"], res)
    return ok


if __name__ == "__main__":
    print("INE 可用年份:", sorted({c["year"] for c in ine_catalog() if c["kind"] == "futures"}))

## 5. 郑商所 CZCE

### 年度包的 URL 分三段，不是一个规律

从历史行情下载页实际抓下来的链接：

| 类别 | 年份 | URL |
|---|---|---|
| 期货 | 2010–2014 | `/cn/exchange/datahistory{年}.zip` |
| 期货 | 2015–2019 | `/cn/DFSStaticFiles/Future/{年}/FutureDataHistory.zip` |
| 期货 | 2020–今 | `/cn/DFSStaticFiles/Future/{年}/ALLFUTURES{年}.zip` |
| 期权 | 2017–2019 | `/cn/DFSStaticFiles/Option/{年}/OptionDataHistory.zip` |
| 期权 | 2020–今 | `/cn/DFSStaticFiles/Option/{年}/ALLOPTIONS{年}.zip` |

网上流传的「`ALLFUTURES{年}.zip` 一路通到底」是**错的**，2019 及以前那样拼会 404。
zip 内容也随年份变：2019 及以前是一整个 txt，2020 起按品种拆开。

年度包走 Chrome；每日 txt `requests` 直连就行（期货最早约 2015-11，期权 2017-04-19 起）。

In [ ]:
CZCE_HOST = "https://www.czce.com.cn"
CZCE_REF  = CZCE_HOST + "/cn/jysj/lshqxz/H077003019index_1.htm"
CZCE_DIR  = ROOT / "czce"


def czce_year_url(year, kind):
    """kind: 'FUTURES' | 'OPTIONS'。没有的年份返回 None。"""
    if kind == "FUTURES":
        if year <= 2014:
            return "%s/cn/exchange/datahistory%d.zip" % (CZCE_HOST, year)
        if year <= 2019:
            return "%s/cn/DFSStaticFiles/Future/%d/FutureDataHistory.zip" % (CZCE_HOST, year)
        return "%s/cn/DFSStaticFiles/Future/%d/ALLFUTURES%d.zip" % (CZCE_HOST, year, year)
    if year < 2017:                       # 白糖期权 2017-04-19 才上市
        return None
    if year <= 2019:
        return "%s/cn/DFSStaticFiles/Option/%d/OptionDataHistory.zip" % (CZCE_HOST, year)
    return "%s/cn/DFSStaticFiles/Option/%d/ALLOPTIONS%d.zip" % (CZCE_HOST, year, year)


def czce_download_history(browser, year_from=YEAR_FROM, year_to=YEAR_TO, overwrite=False,
                          options=True):
    browser.warm(CZCE_REF)
    ok = 0
    for year in range(year_from, year_to + 1):
        for kind in (("FUTURES", "OPTIONS") if options else ("FUTURES",)):
            url = czce_year_url(year, kind)
            if url is None:
                continue
            dst = CZCE_DIR / "history" / ("CZCE_%s_%d.zip" % (kind, year))
            ok += log("CZCE", "%s %d" % (kind, year),
                      browser.get(url, dst, min_size=5000, overwrite=_ow(overwrite, year)))
    return ok


def czce_download_day(day, overwrite=True, options=True, futures=True):
    """每日 txt，requests 直连；强制刷新以覆盖盘中半截文件。"""
    s = make_session(referer=CZCE_REF, retries=1)
    ds, y = day.strftime("%Y%m%d"), day.year
    segs = [("Future", "futures", "FutureDataDaily.txt")] if futures else []
    if options:
        segs.append(("Option", "option", "OptionDataDaily.txt"))
    results = {}
    for seg, kind, fname in segs:
        url = "%s/cn/DFSStaticFiles/%s/%d/%s/%s" % (CZCE_HOST, seg, y, ds, fname)
        dst = CZCE_DIR / "daily" / kind / ("%s.txt" % ds)
        res = fetch_to(s, url, dst, min_size=200, overwrite=overwrite, timeout=(8, 45))
        log("CZCE", "%s %s" % (kind, ds), res)
        results[kind] = res
    return results

## 6. 大商所 DCE

老接口 `publicweb/quotesdata/exportDayQuotesChData.html`（网上教程和老版 akshare 还在用）
现在**返回 500，已经废了**。新版是 iframe SPA `/frontend/dcereport/`，
下载接口从它的打包 JS 里挖出来：

```
GET /dcereport/quote/history/download?type={1|2}&year={年}&variety=all&lang=zh
```

`type=1` 期货、`type=2` 期权，年份 2006–今，zip 内每品种一个 xlsx（`a_ftr.xlsx` / `m_opt.xlsx`）。
2010–2016 的期权包能下但只有表头 —— 豆粕期权 2017-03-31 才上市，属正常。

**新版没有日增量接口了**，日更 = 覆盖重拉当年 zip。

In [ ]:
DCE_HOST = "http://www.dce.com.cn"
DCE_REF  = DCE_HOST + "/dce/channel/list/164.html"
DCE_DIR  = ROOT / "dce"


def dce_year_url(year, kind):
    t = "1" if kind == "futures" else "2"
    return ("%s/dcereport/quote/history/download?type=%s&year=%d&variety=all&lang=zh"
            % (DCE_HOST, t, year))


def dce_download_history(browser, year_from=YEAR_FROM, year_to=YEAR_TO, overwrite=False,
                         options=True):
    browser.warm(DCE_REF)
    ok = 0
    for year in range(max(year_from, 2006), year_to + 1):
        for kind in (("futures", "option") if options else ("futures",)):
            dst = DCE_DIR / "history" / ("DCE_%s_%d.zip" % (kind, year))
            ok += log("DCE", "%s %d" % (kind, year),
                      browser.get(dce_year_url(year, kind), dst,
                                  min_size=5000, overwrite=_ow(overwrite, year)))
    return ok


DCE_DAY_URL = DCE_HOST + "/dcereport/publicweb/dailystat/dayQuotes"


def dce_download_day(browser, day, overwrite=False, options=True, futures=True):
    """DCE 新版日 JSON；普通 requests 412，只在已过挑战的页面里 POST。"""
    if not browser.warm(DCE_REF):
        raise RuntimeError("DCE Chrome 挑战未通过")
    results = {}
    for trade_type, kind in (("1", "futures"), ("2", "option")):
        if kind == "futures" and not futures:
            continue
        if kind == "option" and not options:
            continue
        payload = {"contractId": "", "lang": "zh", "optionSeries": "",
                   "statisticsType": "0", "tradeDate": day.strftime("%Y%m%d"),
                   "tradeType": trade_type, "varietyId": "all"}
        dst = DCE_DIR / "daily" / kind / (day.strftime("%Y%m%d") + ".json")
        res = browser.post_json(DCE_DAY_URL, payload, dst, overwrite=overwrite)
        results[kind] = res
        if not log("DCE", "%s %s" % (kind, day.strftime("%Y%m%d")), res):
            raise RuntimeError("DCE %s 日 JSON 下载失败: %s" % (kind, res))
    return results

## 7. 中金所 CFFEX（`requests` 直连）

按**月**打包：`/sj/historysj/{YYYYMM}/zip/{YYYYMM}.zip`，最早 2010-04。
zip 里是 `{YYYYMMDD}_1.csv`，**GBK 编码，期货和期权混在同一个文件里**
（期权合约形如 `HO2401-C-2000`，靠这个区分）。

`_ow` 对 CFFEX 按**月**判断：当月的重下，往月的跳过。

In [ ]:
CFFEX_HOST  = "http://www.cffex.com.cn"
CFFEX_DIR   = ROOT / "cffex"
CFFEX_START = (2010, 4)          # 沪深300股指期货上市


def _month_range(y0, m0, y1, m1):
    y, m = y0, m0
    while (y, m) <= (y1, m1):
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)


def _cffex_one_month(y, m, overwrite=False, daily=False):
    s = make_session(referer=CFFEX_HOST + "/cn/lssjxz.html", retries=1 if daily else 4)
    ym = "%d%02d" % (y, m)
    today = dt.date.today()
    cur = (y, m) >= (today.year, today.month)      # 当月还在追加，必须重下
    dst = CFFEX_DIR / "history" / ("CFFEX_%s.zip" % ym)
    res = fetch_to(s, "%s/sj/historysj/%s/zip/%s.zip" % (CFFEX_HOST, ym, ym), dst,
                   min_size=1000, expect="zip", overwrite=overwrite or cur,
                   timeout=(8, 45) if daily else TIMEOUT)
    log("CFFEX", ym, res)
    return res


def cffex_download_history(year_from=YEAR_FROM, year_to=YEAR_TO, overwrite=False):
    start = max((year_from, 1), CFFEX_START)
    today = dt.date.today()
    end = (year_to, 12) if year_to < today.year else (today.year, today.month)
    ok = 0
    for y, m in _month_range(start[0], start[1], end[0], end[1]):
        ok += not _cffex_one_month(y, m, overwrite=overwrite).startswith(BAD_PREFIXES)
    return ok


def cffex_download_month_refresh(day=None):
    day = day or dt.date.today()
    return _cffex_one_month(day.year, day.month, overwrite=True, daily=True)

## 8. 广期所 GFEX（`requests` 直连）

2022-12-22 才开业。下载 URL 拼法（从 `hqsj_lshqxz.js` 读出来）是
`/gfex/gfexfile/history/ + 品种大写 + type + 年份 + . + 后缀`，
`type` 取**字面量** `FUTURES` / `OPTIONS`（传 `1`/`2` 清单接口会返回空数组）。

`ALLOPTIONS{年}.csv` 很大（2025 约 59 MB），慢是正常的。

In [ ]:
GFEX_HOST = "http://www.gfex.com.cn"
GFEX_REF  = GFEX_HOST + "/gfex/lshq/lshqxz_new.shtml"
GFEX_DIR  = ROOT / "gfex"


def gfex_fileall(kind="FUTURES", filetype="csv"):
    s = make_session(referer=GFEX_REF)
    s.headers.update({"X-Requested-With": "XMLHttpRequest",
                      "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"})
    r = s.post(GFEX_HOST + "/u/interfacesWebFile/loadList_fileall",
               data={"type": kind, "filetype": filetype}, timeout=60)
    return r.json().get("data", []) or []


def gfex_download_history(year_from=YEAR_FROM, year_to=YEAR_TO, filetype="csv", overwrite=False,
                          options=True, futures=True, return_results=False, retries=4,
                          request_timeout=TIMEOUT):
    s = make_session(referer=GFEX_REF, retries=retries)
    ok = 0
    results = {}
    kinds = []
    if futures:
        kinds.append("FUTURES")
    if options:
        kinds.append("OPTIONS")
    for kind in kinds:
        for it in gfex_fileall(kind, filetype):
            year = int(it["year"])
            if not (year_from <= year <= year_to):
                continue
            dst = GFEX_DIR / "history" / ("GFEX_%s_%d.%s" % (kind, year, filetype))
            res = fetch_to(s, GFEX_HOST + "/gfex/gfexfile/history/" + it["filename"],
                           dst, min_size=1000, overwrite=_ow(overwrite, year),
                           timeout=request_timeout)
            ok += log("GFEX", "%s %d" % (kind, year), res)
            results[kind] = res
    return results if return_results else ok


def gfex_download_year_refresh(year=None, filetype="csv", options=True, futures=True,
                               return_results=False):
    year = year or dt.date.today().year
    return gfex_download_history(year, year, filetype=filetype, overwrite=True,
                                 options=options, futures=futures,
                                 return_results=return_results, retries=1,
                                 request_timeout=(8, 45))

## 9. 一键全量下载

四家直连（SHFE / INE / CFFEX / GFEX）+ 两家走同一个 Chrome 实例（CZCE / DCE）。

**关于重复下载**：往年的包只要本地有就 `skip`，不会重下；**只有当年那份会强制刷新**，
因为交易所还在往里追加。所以每天重跑这个 cell 是安全的，代价只是重下当年的几个包。

第一次全量约 **1.3 GB**，中断了直接重跑，已下好的自动跳过。

In [ ]:
def download_all(year_from=YEAR_FROM, year_to=YEAR_TO, overwrite=False, only=None):
    """only=['shfe','ine'] 之类可以只跑其中几家。"""
    t0, summary = time.time(), {}
    want = lambda n: (not only) or n in only

    for name, fn in (("shfe",  shfe_download_history),
                     ("ine",   ine_download_history),
                     ("cffex", cffex_download_history),
                     ("gfex",  gfex_download_history)):
        if not want(name):
            continue
        print("\n" + "=" * 66 + "\n>>> %s (requests)\n" % name.upper() + "=" * 66)
        try:
            summary[name] = fn(year_from, year_to, overwrite=overwrite)
        except Exception as e:
            summary[name] = "EXCEPTION %s: %s" % (type(e).__name__, e); print("!!", summary[name])

    if want("czce") or want("dce"):
        print("\n" + "=" * 66 + "\n>>> CZCE / DCE (真实 Chrome)\n" + "=" * 66)
        with ChromeDL(CHROME_WORK) as br:
            for name, fn in (("czce", czce_download_history), ("dce", dce_download_history)):
                if not want(name):
                    continue
                print("\n--- %s ---" % name.upper())
                try:
                    summary[name] = fn(br, year_from, year_to, overwrite=overwrite)
                except Exception as e:
                    summary[name] = "EXCEPTION %s: %s" % (type(e).__name__, e); print("!!", summary[name])

    print("\n完成，用时 %.0fs" % (time.time() - t0)); print(summary)
    return summary


# download_all()
# download_all(only=["shfe", "ine", "cffex", "gfex"])   # 不需要 Chrome 的四家

## 10. 每日增量

交易日结算后跑，建议 17:00 之后。非交易日会看到一片 404，正常。

In [ ]:
def download_daily(day=None, options=True):
    day = day or dt.date.today()
    print("### 增量下载 %s%s\n" % (day.strftime("%Y-%m-%d"),
                                "" if options else "（只要期货）"))
    sh = shfe_download_day(day, options=options)   # 这份 dat 是超集，INE 品种也在里面
    cz = czce_download_day(day, overwrite=True, options=options)
    cf = cffex_download_month_refresh(day)
    gf = gfex_download_year_refresh(day.year, options=options, return_results=True)
    # 郑商所日 txt 可直连，并由解析层从文件名补交易日；浏览器只留给大商所。
    with ChromeDL(CHROME_WORK) as br:
        dc = dce_download_day(br, day, overwrite=True, options=options)
    status = {"SHFE": sh.get("futures", "ERR NO_RESULT"),
              "INE": sh.get("futures", "ERR NO_RESULT"),
              "CZCE": cz.get("futures", "ERR NO_RESULT"),
              "DCE": dc.get("futures", "ERR NO_RESULT"),
              "CFFEX": cf, "GFEX": gf.get("FUTURES", "ERR NO_RESULT")}
    errors = {k: v for k, v in status.items() if str(v).startswith(BAD_PREFIXES)}
    if errors:
        raise RuntimeError("日下载存在错误：%s" % errors)
    return status


# download_daily()

# 解析层

把六个来源解析成统一 schema。格式差异很大：

| 来源 | 格式 | 坑 |
|---|---|---|
| SHFE / INE | zip 里的 xls/xlsx，成员名 GBK | 合约列是**合并单元格**要 ffill；期货期权同表；**2021.04 那个文件多一个「品种」前置列**，表头得逐列找 |
| CZCE | 竖线分隔定宽 txt | 2012 及以前 GBK、之后 UTF-8；列名换过三次（品种月份/品种代码/合约代码，空盘量/持仓量） |
| DCE | 每品种一个 xlsx | 数字带千分位逗号；成交额单位是**元** |
| CFFEX | 每天一个 GBK csv | 日期只在文件名里；期货期权混排；早年多一列隐含波动率 |
| GFEX | UTF-8 csv | 第一行是标题不是表头 |

## `series` 列：别把月均价 / MS 合约混进普通品种

2025-2026 出现了带**系列标记**的新合约，命名和普通合约不一样，正则不认就会被整段丢掉：

- `pp2602F` / `l2602F` / `v2602F` —— 大商所**月均价期货**（商品名称写的就是「聚丙烯月均价」），
  在 zip 里是独立的 `pp-F_ftr.xlsx`
- `c2609-MS-C-2160`（玉米/豆粕/豆一/豆油）、`SR603MSC4600`（白糖）—— 带 `MS` 标记的期权，
  **真在交易**，实测最大成交量 36,882 手、持仓 14,106 手

`split_contract` 会把标记拆到 `series` 列，并且**把它拼进 `product`**（`ppf` / `cms`）。
这一步不能省：月均价合约要是和普通 `pp` 算作同一个品种，会一起参与持仓量排序，
主力合约就算错了。

`finalize()` 另外加了告警 —— 丢弃行数超过 50 且占比超 0.1% 就 warn，
避免以后再出现新命名被静默丢掉。

## 两处必要的归一化

1. **成交额统一折算成元**。SHFE/INE/CZCE/CFFEX 原始单位是万元，DCE/GFEX 是元，
   不统一的话合并出来这一列没法用。
2. **加 `double_side` 列标记双边口径**（DCE 全程 True，CZCE 2020-01-01 之前 True，其余 False）。
   成交量和持仓量**保持交易所原值不动**，只打标记 —— 要不要折半是你的决定，
   跟米筐对账时也得先看原值。

In [ ]:
import csv, io, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv

RAW = ROOT

OUT_COLS_FUT = ["trade_date", "exchange", "product", "series", "contract",
                "open", "high", "low", "close", "pre_settle", "settle",
                "volume", "open_interest", "oi_chg", "amount", "double_side"]
OUT_COLS_OPT = OUT_COLS_FUT + ["underlying", "cp_flag", "strike",
                               "delta", "iv", "exercise_vol"]

# 普通: rb2510 / lc2402-C-100000 / SR803C5400 / ag2407C6100 / HO2401-C-2000 / m2403-C-2900
# 带系列标记: pp2602F（大商所月均价期货）/ c2609-MS-C-2160 / SR603MSC4600
# 中间那个 [A-Za-z]{0,3} 就是系列标记，不认它的话这些合约会被整段丢掉。
OPT_RE = re.compile(
    r"^([A-Za-z]{1,3})(\d{3,4})[-_]?([A-Za-z]{0,3}?)[-_]?([CP])[-_]?(\d+(?:\.\d+)?)$", re.I)
FUT_RE = re.compile(r"^([A-Za-z]{1,3})(\d{3,4})([A-Za-z]{0,2})$")


SINGLE_SIDE_FROM = pd.Timestamp("2020-01-01")
# 2020-01-01 起中国商品期货统一改为单边发布，但大商所例外，至今仍发双边。
# 这不是猜的：拿 2019 年末 3 日 / 2020 年初 3 日的全市场成交量比，
# CZCE 0.43、SHFE 0.48、INE 0.54 全部腰斩，DCE 0.875 没动，CFFEX 0.998 一直单边。
# 再跟米筐 daily_contracts 逐行对，也只有 DCE 2020 起是 2.0 倍，其余全程 1.0。
ALWAYS_DOUBLE = {"DCE"}
NEVER_DOUBLE  = {"CFFEX", "GFEX"}      # 中金所一直单边；广期所 2022 才开业


def mark_double_side(df: pd.DataFrame, exchange: str) -> pd.Series:
    """该行的成交量/持仓量是不是双边口径。"""
    if exchange in ALWAYS_DOUBLE:
        return pd.Series(True, index=df.index)
    if exchange in NEVER_DOUBLE:
        return pd.Series(False, index=df.index)
    return pd.to_datetime(df["trade_date"]) < SINGLE_SIDE_FROM


# ----------------------------------------------------------------- 通用工具
def read_excel_fast(buf, **kw):
    """calamine 比 openpyxl/xlrd 快一个数量级（DCE 95MB 包实测 4.2s vs 62s），
    .xls / .xlsx 它都能读。失败再退回默认引擎。"""
    try:
        return pd.read_excel(buf, engine="calamine", **kw)
    except Exception:
        buf.seek(0)
        return pd.read_excel(buf, **kw)


def num(s):
    """把 '1,769,916' / '--' / '' / None 变成 float。"""
    if isinstance(s, pd.Series):
        return pd.to_numeric(
            s.astype(str).str.strip()
             .str.replace(",", "", regex=False)
             .replace({"": None, "-": None, "--": None, "nan": None, "None": None}),
            errors="coerce")
    return pd.to_numeric(str(s).replace(",", ""), errors="coerce")


def split_contract(con: str):
    """返回 (kind, product, underlying, cp, strike, series)。kind: 'F' | 'O' | None

    series 是交割月之后的系列标记，普通合约为空字符串：
      pp2602F        -> series='F'  （大商所聚丙烯"月均价"期货，和 pp2602 是两个不同的合约）
      c2609-MS-C-2160-> series='MS'
      SR603MSC4600   -> series='MS'
    product 会把 series 拼进去（ppf / cms），这样它们不会和普通合约混在一个品种里
    参与主力排序 —— 那会算出完全错误的主力合约。
    """
    c = str(con).strip()
    m = OPT_RE.match(c)
    if m:
        prod, ym, series, cp, strike = m.groups()
        return ("O", (prod + series).lower(), prod + ym, cp.upper(),
                float(strike), series.upper())
    m = FUT_RE.match(c)
    if m:
        prod, ym, series = m.groups()
        return "F", (prod + series).lower(), None, None, np.nan, series.upper()
    return None, None, None, None, np.nan, None


def finalize(df: pd.DataFrame, kind: str) -> pd.DataFrame:
    """补齐缺列、排好列序、清掉解析不出合约的行。"""
    cols = OUT_COLS_OPT if kind == "O" else OUT_COLS_FUT
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    n0 = len(df)
    df = df[df["contract"].notna() & df["product"].notna()]
    dropped = n0 - len(df)
    # 丢几行表尾注释是正常的；成片丢说明出现了没见过的合约命名，必须暴露出来
    if dropped > 50 and dropped / max(n0, 1) > 0.001:
        import warnings as _w
        _w.warn("解析丢弃 %d/%d 行（合约代码无法识别），检查 split_contract 的正则"
                % (dropped, n0), stacklevel=2)
    df = df.copy()
    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
    df = df[df["trade_date"].notna()]
    df["double_side"] = df["double_side"].astype(bool)
    return df[cols].reset_index(drop=True)


def attach_contract_info(df: pd.DataFrame, kind: str) -> pd.DataFrame:
    """按 contract 列拆出 product / underlying / cp / strike，并筛掉另一类。"""
    info = df["contract"].map(split_contract)
    df["_k"]       = info.map(lambda t: t[0])
    df["product"]  = info.map(lambda t: t[1])
    df["series"]   = info.map(lambda t: t[5])
    if kind == "O":
        df["underlying"] = info.map(lambda t: t[2])
        df["cp_flag"]    = info.map(lambda t: t[3])
        df["strike"]     = info.map(lambda t: t[4])
    df = df[df["_k"] == kind].drop(columns=["_k"])
    return df


def gbk_member(n: str) -> str:
    """zip 里 GBK 编码的成员名还原成中文。"""
    try:
        return n.encode("cp437").decode("gbk")
    except Exception:
        return n

In [ ]:
# ----------------------------------------------------------------- SHFE / INE
def _shfe_style_zip(zp: Path, exchange: str, want_ine: bool):
    """上期所和能源中心用的是同一种「所内合约行情报表」表格，期货和期权混在一张表里。

    want_ine: True 只取包内 ine 目录，False 只取非 ine 部分。
    上期所的年度包从 2024 起才把 ine 目录塞进来，2018-2023 完全没有能源中心数据，
    所以 INE 一律从 ine.cn 的独立年度包读，SHFE 这边永远跳过 ine 目录，避免重复计数。
    """
    if not zp.exists():
        return None

    # 解析缓存：这一步是整条流水线最慢的（一年的 xls 要 28s），
    # 而年度包只在月度更新时才变。指纹用文件大小 + mtime，包一变就自动失效。
    st = zp.stat()
    tag = "%s_%s_%d_%d" % (zp.stem, "ine" if want_ine else "main",
                           st.st_size, int(st.st_mtime))
    cp = PARSED / "_parsecache" / (tag + ".pkl")
    if cp.exists():
        try:
            return pd.read_pickle(cp)
        except Exception:
            pass                       # 缓存读不了就当没有，老老实实重解析

    z = zipfile.ZipFile(zp)
    frames = []
    for n in z.namelist():
        if not n.lower().endswith((".xls", ".xlsx")):
            continue
        is_ine = "ine" in gbk_member(n).lower()
        if is_ine != want_ine:
            continue
        raw = read_excel_fast(io.BytesIO(z.read(n)), header=None, dtype=object)
        # 定位表头："合约" 这个单元格。绝大多数文件在第 0 列，但个别月份
        # （实测 2021.04）前面多插了一个"品种"列，所以要逐列找，不能写死 0。
        loc = None
        for c in raw.columns:
            hits = raw.index[raw[c].astype(str).str.strip() == "合约"]
            if len(hits):
                loc = (hits[0], raw.columns.get_loc(c))
                break
        if loc is None:
            continue
        h, c0 = loc
        body = raw.iloc[h + 1:, c0:c0 + 14].copy()
        body.columns = ["contract", "trade_date", "pre_close", "pre_settle", "open",
                        "high", "low", "close", "settle", "ch1", "ch2",
                        "volume", "amount", "open_interest"]
        # 合约列是合并单元格，只在每段第一行有值
        body["contract"] = body["contract"].astype(str).str.strip().replace(
            {"": np.nan, "nan": np.nan, "None": np.nan}).ffill()
        body = body[body["trade_date"].notna()]
        frames.append(body)

    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str).str.strip(),
                                      format="%Y%m%d", errors="coerce")
    for c in ("open", "high", "low", "close", "pre_settle", "settle",
              "volume", "amount", "open_interest"):
        df[c] = num(df[c])
    df["amount"] = df["amount"] * 1e4          # 成交金额单位是万元
    df["exchange"] = exchange
    df["double_side"] = mark_double_side(df, exchange)

    cp.parent.mkdir(parents=True, exist_ok=True)
    for stale in cp.parent.glob("%s_%s_*.pkl" % (zp.stem, "ine" if want_ine else "main")):
        stale.unlink()                 # 同一个包的旧指纹没用了，别越攒越多
    df.to_pickle(cp)
    return df


def parse_shfe_year(year: int, options: bool = True):
    df = _shfe_style_zip(RAW / "shfe" / "history" / "futures" / f"SHFE_futures_{year}.zip",
                         "SHFE", want_ine=False)
    if df is None:
        return None, None
    fut = finalize(attach_contract_info(df.copy(), "F"), "F")
    opt = finalize(attach_contract_info(df.copy(), "O"), "O") if options else None
    return fut, opt


def parse_ine_year(year: int, options: bool = True):
    """能源中心（sc 原油 / lu 低硫燃料油 / nr 20号胶 / bc 国际铜），2018 年起。"""
    df = _shfe_style_zip(RAW / "ine" / "history" / f"INE_futures_{year}.zip",
                         "INE", want_ine=True)
    if df is None:   # 2018 那个包里的成员名不带 ine 前缀
        df = _shfe_style_zip(RAW / "ine" / "history" / f"INE_futures_{year}.zip",
                             "INE", want_ine=False)
    if df is None:
        return None, None
    fut = finalize(attach_contract_info(df.copy(), "F"), "F")
    opt = finalize(attach_contract_info(df.copy(), "O"), "O") if options else None
    return fut, opt

In [ ]:
# ----------------------------------------------------------------- CZCE
def _czce_txt(raw: bytes) -> pd.DataFrame:
    """郑商所是竖线分隔的定宽 txt：第 1 行是标题，第 2 行是表头。老文件 GBK，新文件 UTF-8。"""
    for enc in ("utf-8", "gbk"):
        try:
            txt = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        return pd.DataFrame()

    lines = [l for l in txt.splitlines() if l.strip()]
    hdr_i = next((i for i, l in enumerate(lines[:5])
                  if "交易日期" in l or "合约代码" in l), None)
    if hdr_i is None:
        return pd.DataFrame()
    head = [c.strip() for c in lines[hdr_i].strip().strip("|").split("|")]
    has_trade_date = "交易日期" in head
    rows = []
    for l in lines[hdr_i + 1:]:
        parts = [c.strip() for c in l.strip().strip("|").split("|")]
        if len(parts) < len(head) - 1:
            continue
        first = parts[0].strip()
        if has_trade_date and not first[:4].isdigit():
            continue
        if not has_trade_date and not re.match(r"^[A-Za-z]{1,3}\d", first):
            continue                        # 跳过小计、总计和页尾说明
        rows.append(parts[:len(head)] + [""] * max(0, len(head) - len(parts)))
    return pd.DataFrame(rows, columns=head)


CZCE_RENAME = {"品种月份": "contract", "品种代码": "contract",
               "合约代码": "contract", "交易日期": "trade_date",
               "昨结算": "pre_settle", "今开盘": "open",
               "最高价": "high", "最低价": "low", "今收盘": "close",
               "今结算": "settle", "成交量(手)": "volume",
               "成交量": "volume", "空盘量": "open_interest",
               "持仓量": "open_interest", "增减量": "oi_chg",
               "成交额(万元)": "amount", "DELTA": "delta",
               "隐含波动率": "iv", "行权量": "exercise_vol"}


def _normalize_czce(df: pd.DataFrame, kind: str):
    # 列名随年份变：品种月份/品种代码/合约代码；空盘量/持仓量
    df = df.rename(columns={k: v for k, v in CZCE_RENAME.items() if k in df.columns})
    df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
    for c in ("open", "high", "low", "close", "pre_settle", "settle", "volume",
              "open_interest", "oi_chg", "amount", "delta", "iv", "exercise_vol"):
        if c in df.columns:
            df[c] = num(df[c])
    df["amount"] = df["amount"] * 1e4          # 郑商所成交额单位是万元
    df["exchange"] = "CZCE"
    df["double_side"] = mark_double_side(df, "CZCE")
    return finalize(attach_contract_info(df, kind), kind)


def parse_czce_year(year: int, kind: str):
    """解析年度包；历史回填保留这条路径。"""
    tag = "FUTURES" if kind == "F" else "OPTIONS"
    zp = RAW / "czce" / "history" / f"CZCE_{tag}_{year}.zip"
    if not zp.exists():
        return None
    z = zipfile.ZipFile(zp)
    frames = [_czce_txt(z.read(n)) for n in z.namelist() if n.lower().endswith(".txt")]
    frames = [f for f in frames if len(f)]
    return _normalize_czce(pd.concat(frames, ignore_index=True), kind) if frames else None


def parse_czce_daily(year: int, kind: str):
    """解析直连日文件，并从文件名注入年度包才有的交易日期。"""
    sub = "futures" if kind == "F" else "option"
    frames = []
    for p in sorted((RAW / "czce" / "daily" / sub).glob(f"{year}*.txt")):
        d = _czce_txt(p.read_bytes())
        if len(d):
            d["trade_date"] = pd.to_datetime(p.stem, format="%Y%m%d", errors="coerce")
            frames.append(d)
    return _normalize_czce(pd.concat(frames, ignore_index=True), kind) if frames else None


def merge_daily(base, daily):
    """年度包优先；日文件只补年度包尚未包含的交易日。"""
    frames = [d for d in (base, daily) if d is not None and len(d)]
    if not frames:
        return None
    d = pd.concat(frames, ignore_index=True)
    return (d.drop_duplicates(["trade_date", "exchange", "contract"], keep="first")
             .sort_values(["trade_date", "exchange", "contract"])
             .reset_index(drop=True))

In [ ]:
# ----------------------------------------------------------------- DCE
def parse_dce_year(year: int, kind: str):
    """kind: 'F' -> DCE_futures_{year}.zip, 'O' -> DCE_option_{year}.zip"""
    name = "futures" if kind == "F" else "option"
    zp = RAW / "dce" / "history" / f"DCE_{name}_{year}.zip"
    if not zp.exists():
        return None
    z = zipfile.ZipFile(zp)
    frames = []
    for n in z.namelist():
        if not n.lower().endswith((".xls", ".xlsx")):
            continue
        d = read_excel_fast(io.BytesIO(z.read(n)), dtype=object)
        if len(d):
            frames.append(d)
    if not frames:
        return None                            # 期权 2017 之前是只有表头的空表
    df = pd.concat(frames, ignore_index=True)

    ren = {"合约名称": "contract", "交易日期": "trade_date", "开盘价": "open",
           "最高价": "high", "最低价": "low", "收盘价": "close",
           "前结算价": "pre_settle", "结算价": "settle", "成交量": "volume",
           "持仓量": "open_interest", "持仓量变化": "oi_chg", "成交额": "amount",
           "Delta": "delta", "隐含波动率(%)": "iv", "行权量": "exercise_vol"}
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})
    df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str).str.strip(),
                                      format="%Y%m%d", errors="coerce")
    for c in ("open", "high", "low", "close", "pre_settle", "settle", "volume",
              "open_interest", "oi_chg", "amount", "delta", "iv", "exercise_vol"):
        if c in df.columns:
            df[c] = num(df[c])
    df["exchange"] = "DCE"
    df["double_side"] = mark_double_side(df, "DCE")   # 大商所全程双边
    return finalize(attach_contract_info(df, kind), kind)


def parse_dce_daily(year: int, kind: str):
    """解析页面内 POST 保存的日 JSON，并还原年度包的双边发布口径。"""
    sub = "futures" if kind == "F" else "option"
    frames = []
    for p in sorted((RAW / "dce" / "daily" / sub).glob(f"{year}*.json")):
        try:
            rows = json.loads(p.read_text(encoding="utf-8")).get("data") or []
        except Exception:
            continue
        if rows:
            d = pd.DataFrame(rows)
            d["trade_date"] = pd.to_datetime(p.stem, format="%Y%m%d", errors="coerce")
            frames.append(d)
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True).rename(columns={
        "contractId": "contract", "lastClear": "pre_settle",
        "clearPrice": "settle", "volumn": "volume",
        "openInterest": "open_interest", "diffI": "oi_chg",
        "turnover": "amount", "impliedVolatility": "iv",
        "matchQtySum": "exercise_vol"})
    df["contract"] = df["contract"].astype(str).str.strip()
    for c in ("open", "high", "low", "close", "pre_settle", "settle",
              "volume", "open_interest", "oi_chg", "amount",
              "delta", "iv", "exercise_vol"):
        if c in df.columns:
            df[c] = num(df[c])
    # 新日 JSON 是单边数；项目 schema 保留 DCE 年度包的双边原始发布口径。
    for c in ("volume", "open_interest", "oi_chg"):
        df[c] = df[c] * 2
    df["amount"] = df["amount"] * 2e4
    df["exchange"] = "DCE"
    df["double_side"] = True
    return finalize(attach_contract_info(df, kind), kind)

In [ ]:
# ----------------------------------------------------------------- CFFEX
def _normalize_cffex(df, options=True):
    """中金所全量和目标日解析共用同一套字段标准化。"""
    ren = {"合约代码": "contract", "今开盘": "open", "最高价": "high", "最低价": "low",
           "成交量": "volume", "成交金额": "amount", "持仓量": "open_interest",
           "持仓变化": "oi_chg", "今收盘": "close", "今结算": "settle",
           "前结算": "pre_settle", "Delta": "delta", "隐含波动率(%)": "iv"}
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})
    df["contract"] = df["contract"].astype(str).str.strip()
    df = df[~df["contract"].str.contains("小计|合计|总计", na=False)]
    for c in ("open", "high", "low", "close", "pre_settle", "settle", "volume",
              "open_interest", "oi_chg", "amount", "delta", "iv"):
        if c in df.columns:
            df[c] = num(df[c])
    df["amount"] = df["amount"] * 1e4          # 中金所成交金额单位是万元
    df["exchange"] = "CFFEX"
    df["double_side"] = mark_double_side(df, "CFFEX")
    fut = finalize(attach_contract_info(df.copy(), "F"), "F")
    opt = finalize(attach_contract_info(df.copy(), "O"), "O") if options else None
    return fut, opt


def _parse_cffex_zips(zips, options=True, target_day=None):
    """解析中金所 zip；target_day 非空时在解压前只选精确同日成员。"""
    target_key = as_date(target_day).strftime("%Y%m%d") if target_day is not None else None
    frames = []
    for zp in zips:
        if not Path(zp).exists():
            continue
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if not n.lower().endswith(".csv"):
                    continue
                m = re.search(r"(\d{8})", n)
                if not m or (target_key is not None and m.group(1) != target_key):
                    continue
                try:
                    d = pd.read_csv(io.BytesIO(z.read(n)), encoding="gbk", dtype=object)
                except Exception:
                    continue
                if not len(d):
                    continue
                d["trade_date"] = pd.to_datetime(m.group(1), format="%Y%m%d")
                frames.append(d)
    if not frames:
        return None, None
    return _normalize_cffex(pd.concat(frames, ignore_index=True), options)


def parse_cffex_year(year: int, options: bool = True):
    """中金所按月 zip，每天一个 GBK csv，期货和期权混在一起。"""
    zps = sorted((RAW / "cffex" / "history").glob(f"CFFEX_{year}??.zip"))
    return _parse_cffex_zips(zps, options=options)


def parse_cffex_day(day, options=True):
    day = as_date(day)
    zp = RAW / "cffex" / "history" / ("CFFEX_%s.zip" % day.strftime("%Y%m"))
    return _parse_cffex_zips([zp], options=options, target_day=day)

In [ ]:
# ----------------------------------------------------------------- GFEX
GFEX_RENAME = {"合约代码": "contract", "交易日期": "trade_date",
               "前结算价": "pre_settle", "开盘价": "open",
               "最高价": "high", "最低价": "low", "收盘价": "close",
               "结算价": "settle", "成交量": "volume",
               "持仓量": "open_interest", "持仓量变化": "oi_chg",
               "成交额": "amount", "DELTA": "delta",
               "行权量": "exercise_vol"}


def _normalize_gfex(df, kind):
    if df is None or not len(df):
        return None
    df = df.rename(columns={k: v for k, v in GFEX_RENAME.items() if k in df.columns})
    df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str).str.strip(),
                                      format="%Y%m%d", errors="coerce")
    for c in ("open", "high", "low", "close", "pre_settle", "settle", "volume",
              "open_interest", "oi_chg", "amount", "delta", "exercise_vol"):
        if c in df.columns:
            df[c] = num(df[c])
    df["exchange"] = "GFEX"
    df["double_side"] = mark_double_side(df, "GFEX")
    return finalize(attach_contract_info(df, kind), kind)


def _gfex_day_raw_arrow(path, day):
    """流式扫描年度 CSV，只保留目标日；所有原始列强制字符串。"""
    day_key = as_date(day).strftime("%Y%m%d")
    with Path(path).open("r", encoding="utf-8", newline="") as fh:
        next(fh, None)
        columns = next(csv.reader(fh), None)
    if not columns:
        raise ValueError("GFEX 缺少第二行表头: %s" % path)
    date_col = next((k for k, v in GFEX_RENAME.items() if v == "trade_date"), None)
    if date_col not in columns:
        raise ValueError("GFEX 缺少交易日期列: %s" % columns)
    reader = pacsv.open_csv(
        path,
        read_options=pacsv.ReadOptions(skip_rows=1, autogenerate_column_names=False,
                                        block_size=4 << 20, use_threads=True, encoding="utf8"),
        parse_options=pacsv.ParseOptions(delimiter=",", quote_char='"', double_quote=True,
                                         newlines_in_values=False, ignore_empty_lines=True),
        convert_options=pacsv.ConvertOptions(
            column_types={c: pa.string() for c in columns}, strings_can_be_null=False))
    date_idx = reader.schema.get_field_index(date_col)
    wanted = pa.scalar(day_key, type=pa.string())
    hits = []
    for batch in reader:
        selected = batch.filter(pc.equal(batch.column(date_idx), wanted))
        if len(selected):
            hits.append(selected)
    if not hits:
        return None
    return pa.Table.from_batches(hits, schema=reader.schema).to_pandas().astype(object)


def _gfex_day_raw(path, day):
    """Arrow 解析失败时退到 pandas 分块，不回退到别的交易日。"""
    try:
        return _gfex_day_raw_arrow(path, day)
    except Exception as exc:
        print("  GFEX Arrow 日解析失败，改用 pandas 分块：%s" % type(exc).__name__)
    day_key = as_date(day).strftime("%Y%m%d")
    hits = []
    for chunk in pd.read_csv(path, encoding="utf-8", skiprows=1, dtype=object, chunksize=200000):
        if "交易日期" not in chunk.columns:
            raise ValueError("GFEX 缺少交易日期列")
        hit = chunk[chunk["交易日期"].astype(str).str.strip().eq(day_key)]
        if len(hit):
            hits.append(hit)
    return pd.concat(hits, ignore_index=True) if hits else None


def parse_gfex_year(year: int, kind: str):
    tag = "FUTURES" if kind == "F" else "OPTIONS"
    p = RAW / "gfex" / "history" / f"GFEX_{tag}_{year}.csv"
    if not p.exists():
        return None
    return _normalize_gfex(
        pd.read_csv(p, encoding="utf-8", skiprows=1, dtype=object), kind)


def parse_gfex_day(day, kind):
    day = as_date(day)
    tag = "FUTURES" if kind == "F" else "OPTIONS"
    p = RAW / "gfex" / "history" / ("GFEX_%s_%d.csv" % (tag, day.year))
    if not p.exists():
        return None
    return _normalize_gfex(_gfex_day_raw(p, day), kind)

In [ ]:
# ----------------------------------------------------------------- 汇总
def build_year(year: int, options: bool = True, futures: bool = True):
    """把一年五家的数据合成 (futures_df, options_df)。

    options=False 时只解析期货。全年度期权解析实测 156s、期货只要数秒，
    而日报一行期权数据都不用，日更时关掉它省下来的时间最多。
    """
    futs, opts = [], []

    def add(df, bucket):
        if df is not None and len(df):
            bucket.append(df)

    # 共享原始包，但 options=False 时不要再做期权合约拆分与标准化。
    f, o = parse_shfe_year(year, options=options)
    if futures: add(f, futs)
    if options: add(o, opts)
    f, o = parse_ine_year(year, options=options)
    if futures: add(f, futs)
    if options: add(o, opts)
    f, o = parse_cffex_year(year, options=options)
    if futures: add(f, futs)
    if options: add(o, opts)

    if futures:
        add(merge_daily(parse_czce_year(year, "F"),
                        parse_czce_daily(year, "F")), futs)
        add(merge_daily(parse_dce_year(year, "F"),
                        parse_dce_daily(year, "F")), futs)
        add(parse_gfex_year(year, "F"), futs)
    if options:
        add(merge_daily(parse_czce_year(year, "O"),
                        parse_czce_daily(year, "O")), opts)
        add(merge_daily(parse_dce_year(year, "O"),
                        parse_dce_daily(year, "O")), opts)
        add(parse_gfex_year(year, "O"), opts)

    fut = pd.concat(futs, ignore_index=True) if futs else pd.DataFrame(columns=OUT_COLS_FUT)
    opt = pd.concat(opts, ignore_index=True) if opts else pd.DataFrame(columns=OUT_COLS_OPT)
    for d in (fut, opt):
        if len(d):
            d.sort_values(["trade_date", "exchange", "contract"], inplace=True)
            d.reset_index(drop=True, inplace=True)
    return fut, opt

## 11. 全量解析 → 按年落 parquet

每年一个文件，六个来源已合并，期货和期权分开：

```
<project-root>\parsed\futures\futures_{年}.parquet
<project-root>\parsed\options\options_{年}.parquet
```

全量约 25 分钟（瓶颈是 DCE 那几个近百 MB 的期权 xlsx）。

In [ ]:
FUTURES_EXCHANGES = ("SHFE", "INE", "CZCE", "DCE", "CFFEX", "GFEX")
QUALITY_FIELDS = ("settle", "close", "open_interest", "volume")
FUTURES_ROW_RATIO = 0.95
OPTIONS_ROW_RATIO = 0.80


def exchange_quality(df, expected, min_rows=5, row_ratio=0.8, lookback=20,
                     min_coverage=0.9):
    """按交易所评估单日完整性；行数门槛由 row_ratio 控制。"""
    expected = list(expected)
    missing = [c for c in QUALITY_FIELDS if c not in df.columns]
    if missing:
        raise ValueError("行情 schema 缺少完整性字段：%s" % ", ".join(missing))
    d = df[df["exchange"].isin(expected)].copy()
    if not len(d):
        return pd.DataFrame(columns=["trade_date", "exchange", "rows", "quality_ok"])
    fields = list(QUALITY_FIELDS)
    grouped = d.groupby(["trade_date", "exchange"], sort=True, observed=True)
    rows = grouped.size().rename("rows")
    coverage = grouped[fields].count().div(rows, axis=0)
    coverage.columns = [c + "_coverage" for c in coverage.columns]
    per = (pd.concat([rows, coverage], axis=1).reset_index()
             .sort_values(["exchange", "trade_date"]).reset_index(drop=True))
    per["reference_rows"] = per.groupby("exchange")["rows"].transform(
        lambda s: s.shift(1).rolling(lookback, min_periods=1).median())
    floor = per["reference_rows"].fillna(float(min_rows)) * float(row_ratio)
    per["required_rows"] = np.ceil(np.maximum(float(min_rows), floor)).astype(int)
    ok = per["rows"].ge(per["required_rows"])
    for c in fields:
        ok &= per[c + "_coverage"].ge(min_coverage)
    per["quality_ok"] = ok
    return per


def complete_exchange_dates(df, expected, min_rows=5, row_ratio=0.8, lookback=20,
                            min_coverage=0.9):
    per = exchange_quality(df, expected, min_rows, row_ratio, lookback, min_coverage)
    if not len(per):
        return pd.DatetimeIndex([])
    expected = set(expected)
    good = (per[per["quality_ok"]].groupby("trade_date")["exchange"]
            .agg(lambda x: expected.issubset(set(x))))
    return pd.DatetimeIndex(good[good].index)


def _write_partition(df, path, label, tail_exchanges=("SHFE", "INE"),
                     row_ratio=FUTURES_ROW_RATIO):
    """写入前挡住重复键、来源整所消失和异常缩水，并用原子替换落盘。"""
    path = Path(path)
    keys = ["trade_date", "exchange", "contract"]
    dup = int(df.duplicated(keys).sum())
    if dup:
        raise RuntimeError("%s 有 %d 个重复键，拒绝覆盖 %s" % (label, dup, path.name))
    if path.exists():
        old = pd.read_parquet(path, columns=keys + list(QUALITY_FIELDS))
        old["trade_date"] = pd.to_datetime(old["trade_date"])
        df["trade_date"] = pd.to_datetime(df["trade_date"])
        old_n = old.groupby("exchange").size()
        new_n = df.groupby("exchange").size()
        missing = sorted(set(old_n.index) - set(new_n.index))
        if missing:
            raise RuntimeError("%s 来源消失 %s，拒绝覆盖 %s" % (label, missing, path.name))
        old_q = exchange_quality(old, old_n.index, row_ratio=row_ratio)
        new_q = exchange_quality(df, new_n.index, row_ratio=row_ratio)
        old_good = set(zip(pd.to_datetime(old_q.loc[old_q["quality_ok"], "trade_date"]),
                           old_q.loc[old_q["quality_ok"], "exchange"]))
        new_good = set(zip(pd.to_datetime(new_q.loc[new_q["quality_ok"], "trade_date"]),
                           new_q.loc[new_q["quality_ok"], "exchange"]))
        lost_good = sorted(old_good - new_good)
        if lost_good:
            sample = ["%s:%s" % (ex, stamp.date()) for stamp, ex in lost_good[:5]]
            raise RuntimeError("%s 丢失既有完整交易日 %s" % (label, sample))
        for ex, n in old_n.items():
            floor = 0.70 if ex in tail_exchanges else 0.80
            if new_n.get(ex, 0) < floor * n:
                raise RuntimeError("%s %s 行数异常缩水 %d -> %d"
                                   % (label, ex, n, new_n.get(ex, 0)))
            old_ex, new_ex = old[old["exchange"] == ex], df[df["exchange"] == ex]
            old_max, new_max = old_ex["trade_date"].max(), new_ex["trade_date"].max()
            if new_max < old_max and (old_max, ex) in old_good:
                raise RuntimeError("%s %s 最新完整日倒退 %s -> %s"
                                   % (label, ex, old_max.date(), new_max.date()))
            if new_max >= old_max and (new_max, ex) not in new_good:
                raise RuntimeError("%s %s 最新日 %s 行数或关键字段不完整"
                                   % (label, ex, new_max.date()))
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name("%s.%d.%d.tmp" % (path.name, os.getpid(), time.time_ns()))
    try:
        df.to_parquet(tmp, index=False)
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink()


def _preserve_complete_tail(df, path, exchanges=("SHFE", "INE"),
                            row_ratio=FUTURES_ROW_RATIO):
    """年度包滞后时保留旧分区里已通过质量门的日更尾部。"""
    path = Path(path)
    if not path.exists():
        return df
    old = pd.read_parquet(path)
    old["trade_date"] = pd.to_datetime(old["trade_date"])
    tails = []
    quality = exchange_quality(old, exchanges, row_ratio=row_ratio)
    good = set(zip(pd.to_datetime(quality.loc[quality["quality_ok"], "trade_date"]),
                   quality.loc[quality["quality_ok"], "exchange"]))
    for ex in exchanges:
        old_ex, new_ex = old[old["exchange"] == ex], df[df["exchange"] == ex]
        if not len(old_ex):
            continue
        new_max = pd.to_datetime(new_ex["trade_date"]).max() if len(new_ex) else pd.Timestamp.min
        valid_dates = {stamp for stamp, exchange in good if exchange == ex}
        tail = old_ex[(old_ex["trade_date"] > new_max) & old_ex["trade_date"].isin(valid_dates)]
        if len(tail):
            tails.append(tail)
    if not tails:
        return df
    return (pd.concat([df] + tails, ignore_index=True)
            .drop_duplicates(["trade_date", "exchange", "contract"], keep="first")
            .sort_values(["trade_date", "exchange", "contract"])
            .reset_index(drop=True))


def build_all(years=None, options=True, futures=True):
    years = years or range(YEAR_FROM, YEAR_TO + 1)
    (PARSED / "futures").mkdir(parents=True, exist_ok=True)
    (PARSED / "options").mkdir(parents=True, exist_ok=True)
    rows = []
    for y in years:
        t = time.time()
        fut, opt = build_year(y, options=options, futures=futures)
        fp = PARSED / "futures" / ("futures_%d.parquet" % y)
        op = PARSED / "options" / ("options_%d.parquet" % y)
        if futures:
            fut = _preserve_complete_tail(fut, fp)
            _write_partition(fut, fp, "%d 期货" % y)
        if options:              # 没解析期权时别拿空表把已有的 parquet 覆盖了
            opt = _preserve_complete_tail(opt, op, row_ratio=OPTIONS_ROW_RATIO)
            _write_partition(opt, op, "%d 期权" % y, row_ratio=OPTIONS_ROW_RATIO)
        print("%d  期货 %9s 行  期权 %12s   %5.0fs" %
              (y, format(len(fut), ","),
               (format(len(opt), ",") + " 行") if options else "（跳过）",
               time.time() - t))
        print("     期货来源", fut.groupby("exchange").size().to_dict() if len(fut) else "（未重建）")
        rows.append({"year": y, "futures": len(fut), "options": len(opt)})
    s = pd.DataFrame(rows)
    print("\n期货总行数 %s   期权总行数 %s" %
          (format(s['futures'].sum(), ","), format(s['options'].sum(), ",")))
    return s


# build_all()

## 12. 补 SHFE / INE 年度包的尾巴

上期所和能源中心的**年度包更新滞后**：写这段时包名还是「2026.1月-7月」，
8 月的十几个交易日根本不在包里。别的四家没这个问题。

这一节用日更 `kx{YYYYMMDD}.dat`（JSON）把尾巴补齐并写回 parquet。
SHFE 那份 dat 是超集（`bc/ec/lu/nr/sc` 都在里面），所以只拉 shfe.com.cn 一处，
再按品种拆出 INE，跟年度包的拆法保持一致。

**每次 `build_all()` 之后都要跑一次这个**，否则最近一两个月 SHFE/INE 是空的。

In [ ]:
INE_PRODUCTS = {"sc", "lu", "nr", "bc", "ec"}     # 挂在能源中心的品种


def _dat(seg, day):
    kind = "futures" if seg == "future" else "option"
    local = SHFE_DIR / "daily" / kind / ("kx%s.dat" % day.strftime("%Y%m%d"))
    if local.exists():
        try:
            return json.loads(local.read_text(encoding="utf-8")).get("o_curinstrument") or []
        except Exception:
            pass
    s = make_session(referer=SHFE_HOST + "/reports/tradedata/dailyandweeklydata/", retries=1)
    r = s.get("%s/data/tradedata/%s/dailydata/kx%s.dat" % (SHFE_HOST, seg, day.strftime("%Y%m%d")),
              timeout=(8, 45))
    if r.status_code != 200:
        return None
    try:
        return r.json().get("o_curinstrument") or []
    except Exception:
        return None


def _dat_futures(day):
    rows = _dat("future", day)
    if not rows:
        return None
    d = pd.DataFrame(rows)
    d = d[d["PRODUCTCLASS"].astype(str) == "1"]                    # 6 = TAS 指令，不是独立合约
    d = d[d["PRODUCTGROUPID"].astype(str).str.strip() != ""]
    out = pd.DataFrame({
        "trade_date": pd.Timestamp(day),
        "product":    d["PRODUCTGROUPID"].astype(str).str.strip().str.lower(),
        "contract":   d["PRODUCTGROUPID"].astype(str).str.strip()
                      + d["DELIVERYMONTH"].astype(str).str.strip(),
        "open": num(d["OPENPRICE"]), "high": num(d["HIGHESTPRICE"]),
        "low": num(d["LOWESTPRICE"]), "close": num(d["CLOSEPRICE"]),
        "pre_settle": num(d["PRESETTLEMENTPRICE"]), "settle": num(d["SETTLEMENTPRICE"]),
        "volume": num(d["VOLUME"]), "open_interest": num(d["OPENINTEREST"]),
        "oi_chg": num(d["OPENINTERESTCHG"]),
        "amount": num(d["TURNOVER"]) * 1e4,        # dat 里 TURNOVER 单位是万元
    })
    out["series"] = ""              # dat 拼出来的都是普通合约，没有系列标记
    out["exchange"] = np.where(out["product"].isin(INE_PRODUCTS), "INE", "SHFE")
    out["double_side"] = False
    # dat 里混着 "bc小计" 这类按品种汇总的行（DELIVERYMONTH 是"小计"），带成交量和持仓量，
    # 不过滤就是重复计数。年度包那条路由 finalize() 挡掉，这条路是自己拼的，得单独校验。
    return out[out["contract"].map(lambda c: split_contract(c)[0] == "F")].reindex(columns=OUT_COLS_FUT)


def _dat_options(day):
    rows = _dat("option", day)
    if not rows:
        return None
    d = pd.DataFrame(rows)
    d = d[d["INSTRUMENTID"].astype(str).str.strip() != ""]
    con = d["INSTRUMENTID"].astype(str).str.strip()
    out = pd.DataFrame({
        "trade_date": pd.Timestamp(day),
        "product":    con.map(lambda c: split_contract(c)[1]),
        "series":     con.map(lambda c: split_contract(c)[5]),
        "contract":   con,
        "open": num(d["OPENPRICE"]), "high": num(d["HIGHESTPRICE"]),
        "low": num(d["LOWESTPRICE"]), "close": num(d["CLOSEPRICE"]),
        "pre_settle": num(d["PRESETTLEMENTPRICE"]), "settle": num(d["SETTLEMENTPRICE"]),
        "volume": num(d["VOLUME"]), "open_interest": num(d["OPENINTEREST"]),
        "oi_chg": num(d["OPENINTERESTCHG"]), "amount": num(d["TURNOVER"]) * 1e4,
        "underlying": d["UNDERLYINGINSTRID"].astype(str).str.strip(),
        "cp_flag": con.map(lambda c: split_contract(c)[3]),
        "strike": num(d["STRIKEPRICE"]), "delta": num(d["DELTA"]),
        "iv": np.nan,                                   # 日更 dat 不给隐含波动率
        "exercise_vol": num(d["EXECVOLUME"]),
    })
    out["exchange"] = np.where(out["product"].isin(INE_PRODUCTS), "INE", "SHFE")
    out["double_side"] = False
    return out[out["contract"].map(lambda c: split_contract(c)[0] == "O")].reindex(columns=OUT_COLS_OPT)


DAT_CACHE = PARSED / "_shfe_dat"       # 抓回来的日更 dat，避免每天重抓同一段尾巴


def _cache_path(year, kind):
    return DAT_CACHE / ("shfe_dat_%s_%d.parquet" % (kind, year))


def _cache_load(year, kind):
    p = _cache_path(year, kind)
    if not p.exists():
        return None
    d = pd.read_parquet(p)
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    return d


def _cache_save(year, kind, old, new_rows):
    if not new_rows:
        return
    d = pd.concat(([old] if old is not None else []) + new_rows, ignore_index=True)
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    d = (d.drop_duplicates(subset=["trade_date", "exchange", "contract"], keep="last")
          .sort_values(["trade_date", "exchange", "contract"]).reset_index(drop=True))
    DAT_CACHE.mkdir(parents=True, exist_ok=True)
    path = _cache_path(year, kind)
    tmp = path.with_name(path.name + ".tmp")
    d.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def _complete_shfe_dates(df, year):
    """要求 SHFE/INE 分所达到动态行数与关键字段完整性门槛。"""
    expected = ["SHFE"] + (["INE"] if year >= 2019 else [])
    if df is None or not len(df):
        return set()
    return set(complete_exchange_dates(df, expected, row_ratio=FUTURES_ROW_RATIO))


def fill_shfe_tail(year=None, start=None, end=None, options=True):
    """把年度包里没有、但日更 dat 里有的 SHFE/INE 交易日补进 parquet。"""
    year = year or YEAR_TO
    fp = PARSED / "futures" / ("futures_%d.parquet" % year)
    op = PARSED / "options" / ("options_%d.parquet" % year)
    fut = pd.read_parquet(fp)
    opt = pd.read_parquet(op) if options else None
    fut["trade_date"] = pd.to_datetime(fut["trade_date"])
    if options:
        opt["trade_date"] = pd.to_datetime(opt["trade_date"])

    # 收盘结算前抓的 dat 有成交量和持仓量、但结算价是空的。这种「半截」的日子必须
    # 允许重抓覆盖，否则它会一直算作「已有」，close/settle 永远补不回来。
    sh = fut[fut["exchange"].isin(["SHFE", "INE"])]
    have = _complete_shfe_dates(sh, year)
    stale = set(sh["trade_date"].unique()) - have
    if stale:
        print("  结算价为空、将重抓的日期：", sorted(d.date() for d in stale))
        keep_f = ~(fut["exchange"].isin(["SHFE", "INE"]) & fut["trade_date"].isin(stale))
        fut = fut[keep_f]
        if options:
            keep_o = ~(opt["exchange"].isin(["SHFE", "INE"]) & opt["trade_date"].isin(stale))
            opt = opt[keep_o]
    if start is None:
        if stale:                       # 有半截的日子，就从最早那个开始重抓
            start = min(stale).date()
        elif have:
            start = max(have).date() + dt.timedelta(days=1)
        else:
            start = dt.date(year, 1, 1)
    end   = end or min(dt.date.today(), dt.date(year, 12, 31))

    # 缓存里已经有的日子直接拿来用，不再请求。结算价为空的不算数（还得重抓）。
    cf = _cache_load(year, "F")
    co = _cache_load(year, "O") if options else None
    if cf is not None and len(cf):
        cached_f = _complete_shfe_dates(cf, year) - stale
    else:
        cached_f = set()
    cached_o = (set(co["trade_date"].unique())
                if options and co is not None and len(co) else set())
    cached_o -= stale

    new_f, new_o = [], []        # 本轮真正抓回来的，要写进缓存
    use_f, use_o = [], []        # 本轮要并进 parquet 的（缓存的 + 新抓的）
    hit = 0
    day = start
    while day <= end:
        ts = pd.Timestamp(day)
        if day.weekday() < 5 and ts not in have:
            need_o = options and ts not in cached_o
            if ts in cached_f and not need_o:
                use_f.append(cf[cf["trade_date"] == ts])
                if options and co is not None and ts in cached_o:
                    use_o.append(co[co["trade_date"] == ts])
                hit += 1
                day += dt.timedelta(days=1)
                continue
            f = cf[cf["trade_date"] == ts] if ts in cached_f else _dat_futures(day)
            o = _dat_options(day) if need_o else (
                co[co["trade_date"] == ts] if co is not None and ts in cached_o else None)
            print("  %s  期货 %5d  期权 %6s" % (day, 0 if f is None else len(f),
                                                "-" if o is None else len(o)))
            if f is not None and len(f):
                probe = pd.concat([fut[fut["exchange"].isin(["SHFE", "INE"])]
                                   ] + use_f + [f], ignore_index=True)
                if ts in _complete_shfe_dates(probe, year):
                    use_f.append(f)
                    if ts not in cached_f:
                        new_f.append(f)
                else:
                    print("  !! %s SHFE/INE 期货行数或关键字段不完整，拒绝合并" % day)
            if o is not None and len(o):
                use_o.append(o)
                if ts not in cached_o:
                    new_o.append(o)
            time.sleep(PAUSE)
        day += dt.timedelta(days=1)

    if hit:
        print("  缓存命中 %d 天，省掉这些天的请求" % hit)
    _cache_save(year, "F", cf, new_f)
    if options:
        _cache_save(year, "O", co, new_o)
    new_f, new_o = use_f, use_o

    if not new_f:
        if stale:
            fut = fut.sort_values(["trade_date", "exchange", "contract"]).reset_index(drop=True)
            _write_partition(fut, fp, "%d SHFE/INE 尾部期货" % year)
            if options:
                opt = opt.sort_values(["trade_date", "exchange", "contract"]).reset_index(drop=True)
                _write_partition(opt, op, "%d SHFE/INE 尾部期权" % year,
                                 row_ratio=OPTIONS_ROW_RATIO)
            print("  已清理目标日之后或结算价为空的半截数据")
        else:
            print("  没有需要补的")
        return
    fut2 = pd.concat([fut] + new_f, ignore_index=True)
    opt2 = (pd.concat([opt] + new_o, ignore_index=True) if new_o else opt) if options else None
    for d in [fut2] + ([opt2] if options else []):
        d["trade_date"] = pd.to_datetime(d["trade_date"])
        d.drop_duplicates(subset=["trade_date", "exchange", "contract"], keep="first", inplace=True)
        d.sort_values(["trade_date", "exchange", "contract"], inplace=True)
        d.reset_index(drop=True, inplace=True)
    _write_partition(fut2, fp, "%d SHFE/INE 尾部期货" % year)
    if options:
        _write_partition(opt2, op, "%d SHFE/INE 尾部期权" % year,
                         row_ratio=OPTIONS_ROW_RATIO)
    print("\n  期货 %s -> %s   期权 %s -> %s" %
          (format(len(fut), ","), format(len(fut2), ","),
           format(len(opt), ",") if options else "（跳过）",
           format(len(opt2), ",") if options else "（不变）"))


def build_futures_day(day):
    """只解析目标日；用于日报快路径，避免每天重建整个当前年。"""
    day = pd.Timestamp(day)
    cffex, _ = parse_cffex_day(day, options=False)
    parts = [_dat_futures(day.date()), parse_czce_daily(day.year, "F"),
             parse_dce_daily(day.year, "F"), cffex, parse_gfex_day(day, "F")]
    parts = [d[d["trade_date"].eq(day)] for d in parts if d is not None and len(d)]
    parts = [d for d in parts if len(d)]
    if not parts:
        return pd.DataFrame(columns=OUT_COLS_FUT)
    return (pd.concat(parts, ignore_index=True)
            .drop_duplicates(["trade_date", "exchange", "contract"], keep="last")
            .sort_values(["trade_date", "exchange", "contract"])
            .reset_index(drop=True))


def update_futures_day(day):
    """原子替换目标日六所行情；完整旧日复用，半截日先清理再重建。"""
    day = pd.Timestamp(day)
    fp = PARSED / "futures" / ("futures_%d.parquet" % day.year)
    old = (pd.read_parquet(fp) if fp.exists()
           else pd.DataFrame(columns=OUT_COLS_FUT))
    old["trade_date"] = pd.to_datetime(old["trade_date"])
    good = set(complete_exchange_dates(old, FUTURES_EXCHANGES,
                                       row_ratio=FUTURES_ROW_RATIO))
    had_complete = day in good
    future_bad = {stamp for stamp in old["trade_date"].unique()
                  if pd.Timestamp(stamp) > day and pd.Timestamp(stamp) not in good}
    base = old[~old["trade_date"].isin(future_bad)].copy()
    fresh = build_futures_day(day)
    if not len(fresh):
        if had_complete:
            if len(base) != len(old):
                _write_partition(base, fp, "%d 期货半截尾部清理" % day.year)
            print("  %s 新响应为空，保留已验证的完整旧分区" % day.date())
            return base[base["trade_date"].eq(day)]
        if len(base) != len(old):
            _write_partition(base, fp, "%d 期货半截尾部清理" % day.year)
        print("  %s 六所均无目标日行情" % day.date())
        return fresh
    keep = ~base["trade_date"].eq(day)
    merged = (pd.concat([base[keep], fresh], ignore_index=True)
              .drop_duplicates(["trade_date", "exchange", "contract"], keep="last")
              .sort_values(["trade_date", "exchange", "contract"])
              .reset_index(drop=True))
    if day not in complete_exchange_dates(merged, FUTURES_EXCHANGES,
                                               row_ratio=FUTURES_ROW_RATIO):
        counts = fresh.groupby("exchange").size().to_dict()
        raise RuntimeError("%s 目标日期货不完整：%s" % (day.date(), counts))
    _write_partition(merged, fp, "%d 目标日期货" % day.year)
    return fresh


OPTION_LAUNCH = {"DCE": "2017-03-31", "CZCE": "2017-04-19",
                 "SHFE": "2018-09-21", "CFFEX": "2019-12-23",
                 "INE": "2021-06-21", "GFEX": "2022-12-22"}


def option_expected_exchanges(day):
    day = as_date(day)
    return [ex for ex, launch in OPTION_LAUNCH.items() if day >= pd.Timestamp(launch)]


def build_options_day(day):
    """只解析目标日期权，避免邮件后重建整个当前年。"""
    day = as_date(day)
    _, cffex = parse_cffex_day(day, options=True)
    parts = [_dat_options(day.date()), parse_czce_daily(day.year, "O"),
             parse_dce_daily(day.year, "O"), cffex, parse_gfex_day(day, "O")]
    parts = [d[d["trade_date"].eq(day)] for d in parts if d is not None and len(d)]
    parts = [d for d in parts if len(d)]
    if not parts:
        return pd.DataFrame(columns=OUT_COLS_OPT)
    return (pd.concat(parts, ignore_index=True)
            .drop_duplicates(["trade_date", "exchange", "contract"], keep="last")
            .sort_values(["trade_date", "exchange", "contract"])
            .reset_index(drop=True))


def update_options_day(day):
    """原子替换目标日六所期权；已有完整日直接复用。"""
    day = as_date(day)
    op = PARSED / "options" / ("options_%d.parquet" % day.year)
    old = (pd.read_parquet(op) if op.exists()
           else pd.DataFrame(columns=OUT_COLS_OPT))
    old["trade_date"] = pd.to_datetime(old["trade_date"])
    expected = option_expected_exchanges(day)
    quality = exchange_quality(old, expected, row_ratio=OPTIONS_ROW_RATIO)
    complete_exchanges = set(quality.loc[
        quality["trade_date"].eq(day) & quality["quality_ok"], "exchange"])
    missing_exchanges = sorted(set(expected) - complete_exchanges)
    if not missing_exchanges:
        print("  %s 期权目标日已完整，复用现有分区" % day.date())
        return old[old["trade_date"].eq(day)]
    print("  %s 只补期权缺失交易所 %s" % (day.date(), missing_exchanges))
    fresh = build_options_day(day)
    if not len(fresh):
        raise RuntimeError("%s 没有可用的目标日期权" % day.date())
    preserved = old[old["trade_date"].eq(day) &
                    old["exchange"].isin(complete_exchanges)]
    replacement = fresh[fresh["exchange"].isin(missing_exchanges)]
    target_rows = pd.concat([preserved, replacement], ignore_index=True)
    keep = ~old["trade_date"].eq(day)
    merged = (pd.concat([old[keep], target_rows], ignore_index=True)
              .drop_duplicates(["trade_date", "exchange", "contract"], keep="last")
              .sort_values(["trade_date", "exchange", "contract"])
              .reset_index(drop=True))
    if day not in complete_exchange_dates(merged, expected, row_ratio=OPTIONS_ROW_RATIO):
        counts = target_rows.groupby("exchange").size().to_dict()
        raise RuntimeError("%s 目标日期权不完整：%s" % (day.date(), counts))
    _write_partition(merged, op, "%d 目标日期权" % day.year,
                     row_ratio=OPTIONS_ROW_RATIO)
    sh = replacement[replacement["exchange"].isin(["SHFE", "INE"])]
    if len(sh):
        _cache_save(day.year, "O", _cache_load(day.year, "O"), [sh])
    return target_rows


def fill_shfe_options_day(day):
    """只补目标日 SHFE/INE 期权；不读取、更不覆盖期货分区。"""
    day = as_date(day)
    op = PARSED / "options" / ("options_%d.parquet" % day.year)
    opt = pd.read_parquet(op)
    opt["trade_date"] = pd.to_datetime(opt["trade_date"])
    expected = [ex for ex in ("SHFE", "INE")
                if ex in option_expected_exchanges(day)]
    fresh = _dat_options(day.date())
    if fresh is None or not len(fresh):
        if day in complete_exchange_dates(opt, expected, row_ratio=OPTIONS_ROW_RATIO):
            print("  SHFE/INE 期权目标日已完整，复用现有分区")
            return opt
        raise RuntimeError("%s SHFE/INE 期权没有可用数据" % day.date())
    keep = ~(opt["exchange"].isin(expected) & opt["trade_date"].eq(day))
    merged = pd.concat([opt[keep], fresh], ignore_index=True)
    merged = (merged.drop_duplicates(["trade_date", "exchange", "contract"], keep="last")
                    .sort_values(["trade_date", "exchange", "contract"])
                    .reset_index(drop=True))
    if day not in complete_exchange_dates(merged, expected, row_ratio=OPTIONS_ROW_RATIO):
        raise RuntimeError("%s SHFE/INE 期权行数或关键字段不完整" % day.date())
    _write_partition(merged, op, "%d SHFE/INE 目标日期权" % day.year,
                     row_ratio=OPTIONS_ROW_RATIO)
    co = _cache_load(day.year, "O")
    _cache_save(day.year, "O", co, [fresh])
    return merged


def validate_options_target(day):
    day = as_date(day)
    op = PARSED / "options" / ("options_%d.parquet" % day.year)
    opt = pd.read_parquet(op)
    opt["trade_date"] = pd.to_datetime(opt["trade_date"])
    expected = option_expected_exchanges(day)
    if day not in complete_exchange_dates(opt, expected, row_ratio=OPTIONS_ROW_RATIO):
        q = exchange_quality(opt[opt["trade_date"].eq(day)], expected)
        counts = q.set_index("exchange")["rows"].to_dict() if len(q) else {}
        raise RuntimeError("%s 期权六所不完整：%s" % (day.date(), counts))
    return opt[opt["trade_date"].eq(day)].groupby("exchange").size().to_dict()


def refresh_options(day=None):
    """邮件后置期权链：只下载/写期权；CFFEX 月包虽混合，但不覆盖期货。"""
    day = as_date(day)
    if day is None:
        day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    day = day.date()
    t0 = time.time()
    sh = shfe_download_day(day, overwrite=True, options=True, futures=False)
    cz = czce_download_day(day, overwrite=True, options=True, futures=False)
    cf = cffex_download_month_refresh(day)
    gf = gfex_download_year_refresh(day.year, options=True, futures=False, return_results=True)
    with ChromeDL(CHROME_WORK) as br:
        dc = dce_download_day(br, day, overwrite=True, options=True, futures=False)
    status = {"SHFE/INE": sh.get("option", "ERR NO_RESULT"),
              "CZCE": cz.get("option", "ERR NO_RESULT"),
              "DCE": dc.get("option", "ERR NO_RESULT"),
              "CFFEX": cf, "GFEX": gf.get("OPTIONS", "ERR NO_RESULT")}
    errors = {k: v for k, v in status.items() if str(v).startswith(BAD_PREFIXES)}
    if errors:
        raise RuntimeError("期权下载存在错误：%s" % errors)
    update_options_day(day)
    counts = validate_options_target(day)
    print("期权刷新完成，用时 %.1fs；目标日各所行数 %s" % (time.time() - t0, counts))
    return {"downloads": status, "rows": counts}


# fill_shfe_tail() / refresh_options()

## 13. 覆盖审计

跑完 `build_all()` + `fill_shfe_tail()` 之后用这个体检。

判断标准：SHFE / CZCE / DCE 三家 2010 至今**交易日数应该完全相等**；
CFFEX / GFEX / INE 只在各自上市日之后才该有数据。

In [ ]:
def audit():
    for kind, sub in (("期货", "futures"), ("期权", "options")):
        frames = [pd.read_parquet(p, columns=["trade_date", "exchange"])
                  for p in sorted((PARSED / sub).glob("*.parquet"))]
        frames = [f for f in frames if len(f)]
        if not frames:
            continue
        a = pd.concat(frames, ignore_index=True)
        print("\n--- %s ---" % kind)
        for ex, r in a.groupby("exchange")["trade_date"].agg(["min", "max", "nunique", "size"]).iterrows():
            print("  %-6s %s → %s   %5d 交易日  %12s 行"
                  % (ex, r["min"].date(), r["max"].date(), r["nunique"], format(r["size"], ",")))
        if sub == "futures":
            a["y"] = a["trade_date"].dt.year
            allday = a.groupby("y")["trade_date"].apply(set)
            print("\n  各所相对全市场交易日的缺失（上市前的空档属正常）:")
            for ex, g in a.groupby("exchange"):
                miss = [(y, len(allday[y] - set(dd)))
                        for y, dd in g.groupby("y")["trade_date"] if allday[y] - set(dd)]
                print("    %-6s %s" % (ex, miss or "无缺失"))


# audit()

# 跟米筐对账

## 该跟哪份比

你手上那几份米筐数据是三个不同层级，**只有 `daily_contracts` 能用来对账**：

| 路径 | 是什么 | 用途 |
|---|---|---|
| `主力期货数据\futures_20100101_20260630\daily_contracts\{年}\` | **逐合约日线**，index=(order_book_id, date) | ✅ 对账就用这份 |
| `futures_data\{品种}.{所}\{年}.parquet` | 主力连续，`code=RB.SHF` + `main_code` | ❌ 拼接后的连续序列 |
| `futures_20100101_20260703\daily_88\` | 88 主力连续（`A88`） | ❌ 同上 |
| `rule0/rule1_5m_and_mapping\dominant_mapping_*.parquet` | 每日每品种的主力合约 | ✅ 校验主力算法（不是对账） |

88 和 `futures_data` 是把不同合约首尾拼起来的连续序列，跟逐合约原始行情不是一个东西。

## 实测结果（2010–2026 全量）

**行数：2011–2025 逐行完全匹配，双向零缺失。** 残差只有两处，共 108 行，
全是零成交零持仓的挂牌合约（au1101、wr1908 这种），交易所报表里有行、米筐不收：
2010 年 96 行、2018 年 12 行。2026 我方多 30,549 行是因为米筐这份只到 2026-06-30。

我方额外覆盖**中金所金融期货**（IC/IF/IH/IM/T/TF/TL/TS），米筐这份 `daily_contracts` 不含。

| 字段 | 一致率 | 差异原因 |
|---|---|---|
| `settle` / `pre_settle` | ~100% | 唯一例外是 2013 年 CZCE 的 TC 动力煤 584 行恒定差 0.4 元，郑商所当年文件就那样发的 |
| `close` | 83–98% | **全部是无成交合约**：交易所原始给 0，米筐用结算价填充 |
| `open/high/low` | 同上 | 另有部分行交易所原始就是空白 |
| `volume` / `open_interest` | 2019 前 ~100%，2020 起 ~72% | 全部由 DCE 双边造成，见下 |
| `amount` | 折半后 DCE/GFEX 100%，SHFE/INE 88–92% | CZCE/SHFE/INE 原始单位是"万元"只有 2 位小数，×1e4 有 ≤5 元舍入残差 |

## 双边口径：`double_side` 是标定出来的，不是猜的

拿 2019 年末 3 日 / 2020 年初 3 日的全市场成交量比：

```
CZCE 0.43   SHFE 0.48   INE 0.54   ← 腰斩，2020-01-01 起改单边发布
DCE  0.875  CFFEX 0.998             ← 没变
```

再跟米筐逐行比，也只有 **DCE 从 2020 起是 2.0 倍**，其余全程 1.0。两条证据一致，所以：

- **DCE 全程双边**（至今仍发双边，米筐把它折成单边）
- **SHFE / INE / CZCE 2020-01-01 之前双边，之后单边**
- **CFFEX / GFEX 全程单边**

⚠️ 反过来说，**米筐自己的数据在 2020 年初对 CZCE/SHFE/INE 也是有跳变的**（它只归一了 DCE）。
做长周期成交量、持仓量因子时这是个断点，别当成真实的市场变化。

## CZCE 成交量差异：是米筐做了过滤，不是我们解析错

2024–2026 有 5,311 行（3.8%）CZCE 成交量对不上，**我方永远比米筐大**，
73% 集中在交割月前 0–2 个月，几乎每个交易日都有（598/601 天，每天约 9/235 个合约）。

三方交叉验证下来，问题不在我们这边：

1. **年度包 == 交易所日报**（`FutureDataDaily.txt`，独立发布渠道），逐合约 100% 一致
2. **加总 == 交易所日报里的「总计」行**，三天实测分毫不差：

   | 交易日 | 我方合计 | 交易所「总计」 | 米筐合计 |
   |---|---|---|---|
   | 2026-06-30 | 11,528,177 | **11,528,177** | 11,526,558 |
   | 2025-04-11 | 10,811,052 | **10,811,052** | 9,945,459 |
   | 2025-12-11 | 11,836,728 | **11,836,728** | 10,972,898 |

   持仓量同样对上（2026-06-30：14,752,674 = 14,752,674）

3. 多出来那部分量的隐含成交价 ≈ 当日结算价（中位 0.9994），是正常价格成交，不是期转现

所以我方 = 郑商所官方发布值，米筐对 CZCE 另做了过滤（大概率剔除套利/组合指令腿，
具体规则只有米筐知道）。**做基于交易所口径的报告就用我方这份。**

## 主力算法校验（2024 年，17,299 个「日×品种」）

- **对 rule0 一致率 97.92%** —— 差的 359 个集中在 BB 胶合板(164)、RS 油菜籽(42) 这些僵尸品种
- 对 rule1 一致率 88.46% —— 差的 1,997 个里 JR/LR/PM/RI/WH/ZC 各 242 天（整年不一致），
  这六个都是零成交品种，rule1 应该另有流动性过滤

**rule0 是跟本 notebook 算法最接近的口径。**

In [ ]:
# 米筐逐合约日线所在目录（不是 88、也不是 futures_data，那两个是主力连续）
RQ = Path("D:/Kline/期货/主力期货数据/futures_20100101_20260630/daily_contracts")


def to_rq_id(contract, trade_date):
    """把交易所合约代码转成米筐 order_book_id：字母大写 + 4 位交割月。

    郑商所是 3 位月份（CF401），米筐统一 4 位（CF2401），得按交易日补出十位年份。
    带系列标记的月均价合约米筐也有，只是全大写：pp2602F -> PP2602F。
    """
    c = str(contract).strip()
    m = re.fullmatch(r"([A-Za-z]{1,3})(\d{3,4})([A-Za-z]{0,2})", c)
    if not m:
        return None
    prod, ym, series = m.group(1).upper(), m.group(2), m.group(3).upper()
    if len(ym) == 3:
        digit, mm = int(ym[0]), ym[1:]
        base = pd.Timestamp(trade_date).year
        cand = (base // 10) * 10 + digit
        if cand < base:
            cand += 10
        ym = f"{cand % 100:02d}{mm}"
    return prod + ym + series


def load_rq(year):
    p = RQ / str(year) / f"futures_contracts_daily_{year}.parquet"
    if not p.exists():
        return None
    d = pd.read_parquet(p).reset_index()
    d.columns = [c.lower() for c in d.columns]
    d = d.rename(columns={"order_book_id": "rq_id", "date": "trade_date",
                          "settlement": "settle", "prev_settlement": "pre_settle",
                          "total_turnover": "amount", "open_interest": "oi"})
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    return d


def recon(year, verbose=True):
    mine = pd.read_parquet(PARSED / "futures" / f"futures_{year}.parquet")
    rq = load_rq(year)
    if rq is None:
        print(f"{year}: 米筐无此年份"); return None

    mine = mine.copy()
    mine["rq_id"] = [to_rq_id(c, d) for c, d in zip(mine["contract"], mine["trade_date"])]

    # 米筐 daily_contracts 不含中金所金融期货，先剔掉再比，否则"我方独有"全是噪声
    rq_prefixes = set(re.match(r"^([A-Za-z]+)", i).group(1) for i in rq["rq_id"].unique())
    mine["prefix"] = mine["rq_id"].str.extract(r"^([A-Za-z]+)")[0]
    in_scope = mine["rq_id"].notna() & mine["prefix"].isin(rq_prefixes)

    m = mine[in_scope]
    j = m.merge(rq, on=["rq_id", "trade_date"], how="outer",
                suffixes=("_me", "_rq"), indicator=True)

    both = j[j["_merge"] == "both"]
    only_me = j[j["_merge"] == "left_only"]
    only_rq = j[j["_merge"] == "right_only"]

    print(f"\n{'='*74}\n{year}   我方(可比范围) {len(m):,} 行 | 米筐 {len(rq):,} 行 | "
          f"匹配 {len(both):,} | 仅我方 {len(only_me):,} | 仅米筐 {len(only_rq):,}")
    print(f"       我方被排除(米筐无此品种) {(~in_scope).sum():,} 行 "
          f"[{sorted(set(mine.loc[~in_scope, 'prefix'].dropna()))[:14]}]")

    if len(only_me) and verbose:
        s = only_me.groupby(only_me["rq_id"].str.extract(r"^([A-Za-z]+)")[0]).size().nlargest(6)
        print("       仅我方 top:", s.to_dict())
    if len(only_rq) and verbose:
        s = only_rq.groupby(only_rq["rq_id"].str.extract(r"^([A-Za-z]+)")[0]).size().nlargest(6)
        print("       仅米筐 top:", s.to_dict())

    # 逐字段比对
    pairs = [("open", "open_me", "open_rq"), ("high", "high_me", "high_rq"),
             ("low", "low_me", "low_rq"), ("close", "close_me", "close_rq"),
             ("settle", "settle_me", "settle_rq"), ("pre_settle", "pre_settle_me", "pre_settle_rq"),
             ("volume", "volume_me", "volume_rq"), ("open_interest", "open_interest", "oi"),
             ("amount", "amount_me", "amount_rq")]
    rows = []
    for name, a, b in pairs:
        if a not in both.columns or b not in both.columns:
            continue
        x, y = both[a].astype(float), both[b].astype(float)
        ok = x.notna() & y.notna()
        if not ok.any():
            continue
        rel = (x[ok] - y[ok]).abs() / y[ok].abs().clip(lower=1e-9)
        rows.append({"字段": name, "可比": int(ok.sum()),
                     "完全一致%": round((x[ok] == y[ok]).mean() * 100, 2),
                     "相对误差<1e-6 %": round((rel < 1e-6).mean() * 100, 2),
                     "中位相对误差": f"{rel.median():.2e}",
                     "最大相对误差": f"{rel.max():.2e}"})
    print(pd.DataFrame(rows).to_string(index=False))

    # 双边口径：按交易所看 volume 的比值
    if len(both):
        both = both.copy()
        both["vol_ratio"] = both["volume_me"].astype(float) / both["volume_rq"].astype(float).replace(0, np.nan)
        both["oi_ratio"]  = both["open_interest"].astype(float) / both["oi"].astype(float).replace(0, np.nan)
        g = both.groupby("exchange")[["vol_ratio", "oi_ratio"]].median().round(4)
        g["行数"] = both.groupby("exchange").size()
        print("\n  成交量/持仓量 我方÷米筐 的中位比值（2.0 = 我方双边、米筐单边）:")
        print(g.to_string())
    return both


# recon(2024)                                  # 单年
# for y in range(2010, 2027): recon(y)         # 全量

## 14. 主力 / 次主力合约标注

用米筐 rule0 的官方定义：

> 当同品种其他合约的持仓量在**收盘后**超过当前主力合约 **1.1 倍**时，
> 从**第二个交易日**开始切换主力合约。每个合约只能做一次主力/次主力，不会重复出现。

实现四条：

1. **1.1 倍触发**。差一点点不换，避免换月拉锯期来回抖。
2. **延后一天生效**。收盘后判断，下一交易日才切。
3. **交割月单调**。只能切到交割月不早于当前的合约 —— 这是「不重复出现」的实现方式。
4. **`min_oi=50` 流动性门槛**（我们自己加的，rule0 没有）。当日全场最大持仓都低于它时
   不标注、也不推进状态。僵尸品种零持仓时排序纯属随机，没这道门槛会把随机结果永久固化
   （实测 BB 胶合板 2024 年因此连错 164 天）。

### 第 3 条为什么不能用「记名单」实现

「每个合约只做一次」最直白的写法是记住所有当过主力的合约代码、永不复用。
**实测这样会烂掉**，而且是随时间累积的烂：

| 年份 | 记名单版 主力=当日持仓最大 | 交割月单调版 |
|---|---|---|
| 2010 | 93.8% | 95.3% |
| 2015 | 95.7% | 96.4% |
| 2020 | 89.5% | 95.6% |
| 2024 | 84.7% | 96.0% |
| 2026 | 83.1% | 94.9% |

两个成因：

- **一次误切永久烧掉一个合约**。某个合约短暂冲高触发切换后，真正的主力就进了黑名单，
  后面几个月持仓再大也当不上。实测 2024-01-02 白糖主力挂在 SR403（持仓 241,400），
  而真正的主力 SR405 有 449,414 —— PTA 更夸张，TA402 vs TA405 差 4.7 倍。
- **郑商所 3 位代码跨十年撞名**。`SR405` 在 2014 年是 2014-05、2024 年是 2024-05，
  同一个字符串，2014 用过一次就把 2024 那个连坐了。

交割月单调没有这两个问题：月份单调本身就蕴含「一个合约只会连续担任一段主力」，
而且真正的主力只要月份不比当前旧，随时可以接管，不存在死锁。

### 实测对比（全历史 2010–2026 标注，一致率在 2024–2026 上比）

| 实现 | 对 rule0 一致 | 主力=当日持仓最大 | 主力<最大的50% | 倒退交割月 |
|---|---|---|---|---|
| 早期：5日均持仓 + 棘轮 | 97.92% | 94.73% | 0.69% | 0 |
| 记名单版 | 88.52% | 89.91% | 5.32% | 145 |
| **交割月单调版（现在）** | **99.51%** | **95.65%** | **0.36%** | **0** |

### 关于米筐自己

rule0 的映射表里 1,203 次换月中有 13 次切到更早交割月（1.08%），
另有 158 个合约「中断后又回来当主力」。所以「每个合约只做一次」在米筐自己的数据里
也不是严格成立的硬约束，更像对正常情况的描述。

另外**不要直接套米筐的 rule0 映射表**：它 2026 年只有 80 个品种，
完全没有中金所（if/ic/im/ih/t/tf/tl/ts），也没有 lf/ppf/vf，
反而多标 8 个全年零持仓的僵尸品种。我们自己算的覆盖 97 个。

In [ ]:
def delivery_key(contract, trade_date):
    """交割月 → 可比较整数（年*12+月）。郑商所 3 位代码按交易日补出十位年份。"""
    m = re.search(r"(\d{3,4})[A-Za-z]{0,2}\s*$", str(contract).strip())
    if not m:
        return -1
    d = m.group(1)
    if len(d) == 4:
        return (2000 + int(d[:2])) * 12 + int(d[2:])
    digit, mm = int(d[0]), int(d[1:])
    # trade_date 传年份整数或日期都行 —— 只用得到年份
    base = trade_date if isinstance(trade_date, (int, np.integer)) else pd.Timestamp(trade_date).year
    cand = (base // 10) * 10 + digit
    if cand < base:
        cand += 10
    return cand * 12 + mm


def tag_dominant(df, min_oi=50, ratio=1.1, date_col="trade_date", prod_col="product",
                 con_col="contract", oi_col="open_interest",
                 initial_state=None, return_state=False):
    """返回 DataFrame[trade_date, product, main, sub]。

    米筐 rule0 的定义：
      当同品种其他合约的持仓量在收盘后超过当前主力合约 1.1 倍时，
      从第二个交易日开始切换主力合约。每个合约只能做一次主力/次主力，不会重复出现。

    实现要点：
    1. **1.1 倍触发**：收盘后才判断，**下一个交易日**生效。差一点点不换，避免来回抖。
    2. **交割月单调**：只能切到交割月不早于当前的合约。这就是「不重复出现」的实现方式
       —— 比记名单安全，不会因为一次误切把好合约永久烧掉。
    3. **次主力永远比主力新**：主力换月时次主力的门槛跟着抬到主力的交割月，
       所以卸任的主力不会退回去当次主力。
    4. **min_oi 门槛**：当日全场最大持仓都低于它时不标注、也不推进状态。
       僵尸品种零持仓时排序纯属随机，没这道门槛会把随机结果永久固化。
    """
    d = df[[date_col, prod_col, con_col, oi_col]].copy()
    d[date_col] = pd.to_datetime(d[date_col])
    # delivery_key 只跟 (合约代码, 交易日年份) 有关，213 万行里不同组合只有几万个。
    # 不缓存的话光这一行就要跑 213 万次正则 + 213 万次 Timestamp 构造。
    _years = d[date_col].dt.year.values
    _contracts = d[con_col].astype(object).to_numpy(copy=False)
    _cache = {}
    _k = np.empty(len(d), dtype=np.int64)
    for _i, (_c, _y) in enumerate(zip(_contracts, _years)):
        _key = (_c, _y)
        _v = _cache.get(_key)
        if _v is None:
            _v = _cache[_key] = delivery_key(_c, _y)
        _k[_i] = _v
    d["_k"] = _k

    rows = []
    state_rows = []
    initial_state = initial_state or {}
    for prod, g in d.groupby(prod_col, sort=False):
        seed = initial_state.get(prod, {})
        main, sub = seed.get("main"), seed.get("sub")
        main_k, sub_k = seed.get("main_k", -1), seed.get("sub_k", -1)
        pend_main, pend_sub = seed.get("pend_main"), seed.get("pend_sub")

        # 逐日切片走 numpy，比每天 groupby 出一个 DataFrame 快一个量级
        g = g.sort_values(date_col, kind="stable")
        _dates = g[date_col].values
        _cons  = g[con_col].astype(object).to_numpy(copy=False)
        _ois   = g[oi_col].astype(float).values
        _ks    = g["_k"].values
        _bnd   = np.flatnonzero(np.r_[True, _dates[1:] != _dates[:-1]])
        _bnd   = np.r_[_bnd, len(_dates)]

        for _b in range(len(_bnd) - 1):
            lo, hi = _bnd[_b], _bnd[_b + 1]
            date = pd.Timestamp(_dates[lo])
            oi = {}
            kk = {}
            for c, v, k in zip(_cons[lo:hi], _ois[lo:hi], _ks[lo:hi]):
                oi[c] = 0.0 if v != v else float(v)
                kk[c] = k
            if not oi:
                continue
            if min_oi and max(oi.values()) < min_oi:
                rows.append({date_col: date, prod_col: prod, "main": None, "sub": None})
                state_rows.append({date_col: date, prod_col: prod, "main": main, "sub": sub,
                                   "main_k": main_k, "sub_k": sub_k,
                                   "pend_main": pend_main, "pend_sub": pend_sub})
                continue

            # 昨日收盘定下的切换，今天生效
            if pend_main is not None and pend_main in oi:
                main, main_k = pend_main, kk[pend_main]
                sub_k = max(sub_k, main_k)      # 次主力门槛跟着抬，旧主力回不来
                if sub == main:
                    sub = None
            pend_main = None
            if pend_sub is not None and pend_sub in oi and pend_sub != main:
                sub, sub_k = pend_sub, kk[pend_sub]
            pend_sub = None

            # 主力摘牌或尚未确立
            if main is None or main not in oi:
                cand = [c for c in oi if kk[c] >= main_k]
                if cand:
                    main = max(cand, key=lambda c: oi[c])
                    main_k = kk[main]
                    sub_k = max(sub_k, main_k)
            if sub is None or sub not in oi or sub == main:
                cand = [c for c in oi if c != main and kk[c] >= sub_k]
                sub = max(cand, key=lambda c: oi[c]) if cand else None
                if sub is not None:
                    sub_k = kk[sub]

            rows.append({date_col: date, prod_col: prod, "main": main, "sub": sub})

            # 收盘后判断下一交易日是否切换
            if main is not None:
                c2 = [c for c in oi if c != main and kk[c] >= main_k
                      and oi[c] > ratio * oi[main]]
                if c2:
                    pend_main = max(c2, key=lambda c: oi[c])
            if sub is not None:
                skip = {main, sub} | ({pend_main} if pend_main else set())
                c2 = [c for c in oi if c not in skip and kk[c] >= sub_k
                      and oi[c] > ratio * oi[sub]]
                if c2:
                    pend_sub = max(c2, key=lambda c: oi[c])
            state_rows.append({date_col: date, prod_col: prod, "main": main, "sub": sub,
                               "main_k": main_k, "sub_k": sub_k,
                               "pend_main": pend_main, "pend_sub": pend_sub})

    tags = (pd.DataFrame(rows).sort_values([prod_col, date_col])
            .reset_index(drop=True))
    if not return_state:
        return tags
    states = (pd.DataFrame(state_rows).sort_values([prod_col, date_col])
              .reset_index(drop=True))
    return tags, states


# --- 自检：在 Notebook 手工运行时显示；daily_job 导入定义时不执行 ---
if __name__ == "__main__":
    _demo = pd.DataFrame([
        ("2025-01-02", "rb", "rb2505", 100), ("2025-01-02", "rb", "rb2510",  90),
        ("2025-01-02", "rb", "rb2601",  20),
        ("2025-01-03", "rb", "rb2505",  80), ("2025-01-03", "rb", "rb2510", 200),
        ("2025-01-03", "rb", "rb2601",  25),
        ("2025-01-06", "rb", "rb2505",  70), ("2025-01-06", "rb", "rb2510", 210),
        ("2025-01-06", "rb", "rb2601",  30),
        ("2025-01-07", "rb", "rb2505", 500), ("2025-01-07", "rb", "rb2510", 210),
        ("2025-01-07", "rb", "rb2601",  40),
    ], columns=["trade_date", "product", "contract", "open_interest"])
    print(tag_dominant(_demo, min_oi=0))
# 01-03 收盘 rb2510(200) > 1.1 x rb2505(80) -> 01-06 起换主力，次主力顺延到 rb2601；
# 01-07 rb2505 持仓反超到 500 也拿不回任何角色（交割月更早）

## 15. 主力 / 次主力落文件

`tag_dominant()` 只返回 `[trade_date, product, main, sub]` 四列合约代码。
做报告要的是**主力合约那根日线**，所以这里把标注 join 回行情，
落成两份**结构完全相同**的文件：

```
<project-root>\dominant\main.parquet  +  main.csv     主力
<project-root>\dominant\sub.parquet   +  sub.csv      次主力
```

列：`trade_date, exchange, product, contract, role` + 完整日线
（`open/high/low/close/pre_settle/settle/volume/open_interest/oi_chg/amount/double_side`）。

⚠️ **棘轮状态必须跨年连续**，所以只能一次性载入全部年份再标注，不能逐年分开跑
（逐年跑等于每年 1 月 1 日把棘轮清零，换月逻辑就废了）。

⚠️ 这是**逐合约日线的拼接，不做价格调整**。换月那天 `close` 会跳空，
直接拿它算收益率是错的 —— 要连续价格得自己按换月日的价差 / 比值接。
米筐的 `futures_data`、`daily_88` 是已经接好的那种，用途不一样。

⚠️ 有 `min_oi` 门槛，僵尸品种（`jr/lr/pm/ri/wh/zc/bb/rs/wr`）会**没有行**，
不是漏了。用之前别假设每个品种每天都有主力。

In [ ]:
DOM = BASE / "dominant"
BAR_COLS = ["open", "high", "low", "close", "pre_settle", "settle",
            "volume", "open_interest", "oi_chg", "amount", "double_side"]


def load_futures(years=None):
    """把 parsed/futures 下的年度 parquet 拼成一张表。"""
    fs = sorted((PARSED / "futures").glob("futures_*.parquet"))
    if years:
        years = set(int(y) for y in years)
        fs = [f for f in fs if int(re.search(r"(\d{4})", f.name).group(1)) in years]
    d = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    return d


def build_dominant(min_oi=50, ratio=1.1, csv=True, fut=None, changed_from=None):
    """写主/次主力；自动链只重算 changed_from 及之后的状态后缀。"""
    DOM.mkdir(parents=True, exist_ok=True)
    provided_fut = fut is not None
    fut = load_futures() if fut is None else fut
    fut["trade_date"] = pd.to_datetime(fut["trade_date"])
    print("载入期货 %s 行，%s ~ %s" % (format(len(fut), ","),
          fut["trade_date"].min().date(), fut["trade_date"].max().date()))

    state_path = DOM / "state.parquet"
    role_paths = {role: DOM / ("%s.parquet" % role) for role in ("main", "sub")}
    before = pd.Timestamp(changed_from) if changed_from is not None else None
    incremental = False
    state_old = None
    initial_state = {}
    work_fut = fut

    if before is not None and state_path.exists() and all(p.exists() for p in role_paths.values()):
        state_old = pd.read_parquet(state_path)
        state_old["trade_date"] = pd.to_datetime(state_old["trade_date"])
        required = {"trade_date", "product", "main", "sub", "main_k", "sub_k",
                    "pend_main", "pend_sub", "state_version", "min_oi", "ratio"}
        params_ok = (required.issubset(state_old.columns) and
                     state_old["state_version"].eq(1).all() and
                     state_old["min_oi"].eq(float(min_oi)).all() and
                     state_old["ratio"].eq(float(ratio)).all())
        prior = state_old[state_old["trade_date"] < before]
        latest_before = fut.loc[fut["trade_date"] < before, "trade_date"].max()
        coverage_ok = (pd.isna(latest_before) or
                       (len(prior) and state_old["trade_date"].max() >= latest_before))
        if params_ok and coverage_ok and len(prior):
            seeds = (prior.sort_values(["product", "trade_date"])
                     .groupby("product", sort=False).tail(1))
            for row in seeds.itertuples(index=False):
                clean = lambda value: None if pd.isna(value) else value
                initial_state[row.product] = {
                    "main": clean(row.main), "sub": clean(row.sub),
                    "main_k": int(row.main_k), "sub_k": int(row.sub_k),
                    "pend_main": clean(row.pend_main),
                    "pend_sub": clean(row.pend_sub),
                }
            work_fut = fut[fut["trade_date"] >= before]
            incremental = len(work_fut) > 0

    # report.load_all() 为出报速度只载入近年数据；首次没有状态快照时，
    # 必须回到全部年度分区全量建状态，不能用近年窗口截断历史主力。
    if before is not None and not incremental and provided_fut:
        fut = load_futures()
        work_fut = fut

    tags, state_new = tag_dominant(
        work_fut, min_oi=min_oi, ratio=ratio,
        initial_state=initial_state if incremental else None, return_state=True)
    if incremental:
        state = pd.concat([state_old[state_old["trade_date"] < before], state_new],
                          ignore_index=True)
        print("增量重算主力状态：%s 起，共 %s 行行情" %
              (before.date(), format(len(work_fut), ",")))
    else:
        state = state_new
    state = (state.drop_duplicates(["trade_date", "product"], keep="last")
             .sort_values(["product", "trade_date"]).reset_index(drop=True))
    state["state_version"] = 1
    state["min_oi"] = float(min_oi)
    state["ratio"] = float(ratio)
    print("本次日×品种 %s 行，有主力 %s" %
          (format(len(tags), ","), format(int(tags["main"].notna().sum()), ",")))

    bar = work_fut.set_index(["trade_date", "contract"])
    bar = bar[~bar.index.duplicated(keep="first")]
    out = {}
    for role in ("main", "sub"):
        t = (tags[tags[role].notna()][["trade_date", "product", role]]
             .rename(columns={role: "contract"}))
        j = t.join(bar[["exchange"] + BAR_COLS], on=["trade_date", "contract"])
        j["role"] = role
        j = j[["trade_date", "exchange", "product", "contract", "role"] + BAR_COLS]
        if incremental:
            old_j = pd.read_parquet(role_paths[role])
            old_j["trade_date"] = pd.to_datetime(old_j["trade_date"])
            j = pd.concat([old_j[old_j["trade_date"] < before], j], ignore_index=True)
        j = j.sort_values(["product", "trade_date"]).reset_index(drop=True)
        f = role_paths[role]
        tmp = f.with_name("%s.%d.%d.tmp" % (f.name, os.getpid(), time.time_ns()))
        try:
            j.to_parquet(tmp, index=False)
            os.replace(tmp, f)
        finally:
            if tmp.exists():
                tmp.unlink()
        if csv:
            csv_path = DOM / ("%s.csv" % role)
            csv_tmp = csv_path.with_name("%s.%d.%d.tmp" %
                                          (csv_path.name, os.getpid(), time.time_ns()))
            try:
                j.to_csv(csv_tmp, index=False, encoding="utf-8-sig")
                os.replace(csv_tmp, csv_path)
            finally:
                if csv_tmp.exists():
                    csv_tmp.unlink()
        print("  %-4s %9s 行 -> %s" % (role, format(len(j), ","), f.name))
        out[role] = j

    state_tmp = state_path.with_name("%s.%d.%d.tmp" %
                                     (state_path.name, os.getpid(), time.time_ns()))
    try:
        state.to_parquet(state_tmp, index=False)
        os.replace(state_tmp, state_path)
    finally:
        if state_tmp.exists():
            state_tmp.unlink()
    out["state"] = state
    return out


# build_dominant()

## 16. 每日一键更新

`download_all()` 也能拿到最新一天（当年的包本来就强制刷新），但它要把六个来源
**整年的包**重下一遍，几百 MB，日常没必要。

`download_daily()` 拉目标日所需原始文件：SHFE/INE 日更 dat、CZCE 日更 txt、
DCE 日 JSON、CFFEX 当月包和 GFEX 当年 CSV。之后只解析并原子替换目标日，
不再日更重建整个年份。`download_daily()` 本身仍然**只下载**，不解析、不标注。

下面这个把后面三步串上，收盘结算后跑这一个 cell 就够了。

In [ ]:
def daily_update(day=None, rebuild_dominant=True, options=False,
                 dominant_csv=True, render_report=True, update_index=True):
    """一键跑完：下载 → 解析当年 → 补 SHFE/INE 尾巴 → 重算主力 → 出 HTML 日报。

    day 怎么写都行：daily_update(20260819) / "2026-08-19" / date 对象。
    **留空默认昨天** —— 当天盘中数据不全（结算价还没出），拿它出报告是半截的；
    要看当天的，收盘结算后（17:00 以后）显式写 daily_update(今天的日期)。
    非交易日会看到一片 404，然后「没有需要补的」，不会写坏任何东西。

    **options 默认 False** —— 日报一行期权数据都不用，而期权是大头：
    全年度期权解析约 156s，而目标日增量只需解析当天需要的原始数据。
    自动任务用 refresh_options(day) 在邮件后刷新并原子合并目标日期权，不覆盖期货分区。
    """
    day = as_date(day)
    if day is None:
        day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    day = day.date()
    t0 = time.time()
    index_error = None

    print("=" * 66 + "\n[1/5] 下载 %s\n" % day.strftime("%Y-%m-%d") + "=" * 66)
    downloads = download_daily(day, options=options)

    print("\n" + "=" * 66 + "\n[2/5] 解析并原子合并目标日\n" + "=" * 66)
    if options:
        build_all([day.year], options=True)
        fill_shfe_tail(day.year, end=day, options=True)
    else:
        update_futures_day(day)

    if rebuild_dominant:
        print("\n" + "=" * 66 + "\n[4/5] 重算主力 / 次主力\n" + "=" * 66)
        build_dominant(csv=dominant_csv)

    if update_index:
        print("\n" + "=" * 66 + "\n[5/5] 更新指数 / 生成 HTML 日报\n" + "=" * 66)
        # 指数优先走上交所官方接口，腾讯备用。两者都失败才跳过。
        for attempt in range(3):
            try:
                download_index(beg=(day - dt.timedelta(days=30)).strftime("%Y%m%d"))
                index_error = None
                break
            except Exception as e:
                index_error = e
                print("  指数下载第 %d 次失败：%s" % (attempt + 1, type(e).__name__))
                time.sleep(3)
        else:
            print("  !! 指数没更新；若目标日有期货行情将拒绝正式发信")

    fp = PARSED / "futures" / ("futures_%d.parquet" % day.year)
    fut = (pd.read_parquet(fp, columns=["trade_date", "exchange"])
           if fp.exists() else pd.DataFrame(columns=["trade_date", "exchange"]))
    fut["trade_date"] = pd.to_datetime(fut["trade_date"])
    target_rows = fut[fut["trade_date"].eq(pd.Timestamp(day))]
    if update_index and index_error is not None and len(target_rows):
        raise RuntimeError("目标日已有期货行情，但指数更新失败") from index_error
    if render_report:
        build_report(day)
    last = fut["trade_date"].max() if len(fut) else None
    print("\n完成，用时 %.0fs。当年最新交易日 %s，当日各所行数 %s" %
          (time.time() - t0, pd.Timestamp(last).date() if last is not None else "无",
           fut[fut["trade_date"] == last].groupby("exchange").size().to_dict()
           if last is not None else {}))
    return {"downloads": downloads, "target_rows": len(target_rows),
            "index_error": index_error}


# daily_update()

## 17. 股指期货的标的指数

做基差要有现货指数。中证指数官网有反爬，本机也没装 akshare / rqdatac，
现改走**上交所官方日 K**（腾讯备用）：免鉴权、字段干净，且避开东财限流。

沪深300 / 中证500 / 中证1000 / 上证50 各 4,039 行，2010-01-04 起，
**和期货交易日一天不差**。落到 `parsed/index/index_daily.parquet`。

In [ ]:
import requests

IDX_DIR = PARSED / "index"
IDX_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
      "Referer": "https://www.sse.com.cn/"}
IDX_SESSION = requests.Session()
IDX_SESSION.trust_env = False          # 指数直连不继承容易误分流的系统代理

# secid 前缀 1 = 沪市。对应关系写死，别靠名字猜。
INDEX = {
    "000300": ("沪海深300", "1.000300", "if"),
    "000905": ("中证500",  "1.000905", "ic"),
    "000852": ("中证1000", "1.000852", "im"),
    "000016": ("上证50",   "1.000016", "ih"),
}
INDEX["000300"] = ("沪深300", "1.000300", "if")

COLS = ["trade_date", "open", "close", "high", "low", "volume", "amount",
        "amplitude", "pct_chg", "chg", "turnover"]


def fetch_sse(index_code, beg="20100101"):
    """上交所官方日 K。32042 是官方 HTTPS 行情端口，443 不提供这项服务。"""
    start = pd.Timestamp(str(beg))
    calendar_days = max(90, (pd.Timestamp.today().normalize() - start).days + 20)
    count = min(7000, int(calendar_days * 0.8) + 40)
    url = "https://yunhq.sse.com.cn:32042/v1/sh1/dayk/%s" % index_code
    r = IDX_SESSION.get(url, params={"select": "date,open,high,low,close,volume",
                                     "begin": -count, "end": -1},
                        headers=IDX_HEADERS, timeout=(5, 15))
    r.raise_for_status()
    rows = r.json().get("kline") or []
    if not rows:
        raise RuntimeError("上交所没拿到 %s" % index_code)
    d = pd.DataFrame(rows, columns=["trade_date", "open", "high", "low",
                                    "close", "volume"])
    d["trade_date"] = pd.to_datetime(d["trade_date"].astype(str), format="%Y%m%d")
    for c in ("open", "close", "high", "low", "volume"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["amount"] = np.nan
    d["amplitude"] = np.nan
    d["pct_chg"] = d["close"].pct_change() * 100
    d["chg"] = d["close"].diff()
    d["turnover"] = np.nan
    return d[d["trade_date"] >= start][COLS].reset_index(drop=True)


def fetch(secid, beg="20100101"):
    u = ("https://push2his.eastmoney.com/api/qt/stock/kline/get"
         "?secid=%s&klt=101&fqt=0&beg=%s&end=20500101"
         "&fields1=f1,f2,f3,f4,f5,f6&fields2=f51,f52,f53,f54,f55,f56,f57,f58,f59,f60,f61"
         % (secid, beg))
    # 先直连。东财在国内，走代理反而容易被分流规则送到不通的出口上去
    # （实测本机代理端口在听，但 CONNECT 直接被断开）。直连失败再退回代理。
    err = None
    for proxies in ({"http": None, "https": None}, None):
        try:
            j = requests.get(u, headers=IDX_HEADERS, timeout=60,
                             proxies=proxies).json()["data"]
            break
        except Exception as e:
            err = e
    else:
        raise err
    d = pd.DataFrame([k.split(",") for k in j["klines"]], columns=COLS)
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    for c in COLS[1:]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d


# ---- 备用源：腾讯日 K ----
# 东财对同一 IP 限流时，push2* 上所有 /api/ 请求都会被直接断开（看起来像网络问题，
# 其实是被挡了）。腾讯的收盘价跟东财实测逐日完全一致。
TX_URL  = "https://web.ifzq.gtimg.cn/appstock/app/fqkline/get"
TX_CODE = {"000300": "sh000300", "000905": "sh000905",
           "000852": "sh000852", "000016": "sh000016"}


def _tx_page(code, beg, end):
    r = IDX_SESSION.get(TX_URL, params={"param": "%s,day,%s,%s,320,qfq" % (code, beg, end)},
                     headers={"User-Agent": IDX_HEADERS["User-Agent"]},
                     timeout=(5, 15))
    j = r.json().get("data")
    if not isinstance(j, dict) or code not in j:
        return []
    d = j[code]
    return d.get("day") or d.get("qfqday") or []


def fetch_tencent(index_code, beg="20100101"):
    """单次最多约 320 行，按半年分页。
    腾讯只给开高低收和成交量，成交额那几列留空；涨跌幅用相邻收盘价自己算
    （跟东财官方值实测差 ≤0.005 个百分点，纯粹是四舍五入的位数不同）。"""
    code = TX_CODE[index_code]
    y0 = int(str(beg)[:4])
    rows = []
    for y in range(y0, dt.date.today().year + 1):
        for b, e in (("%d-01-01" % y, "%d-06-30" % y), ("%d-07-01" % y, "%d-12-31" % y)):
            if pd.Timestamp(e) < pd.Timestamp(str(beg)):
                continue
            rows += _tx_page(code, b, e)
            time.sleep(0.3)
    if not rows:
        raise RuntimeError("腾讯也没拿到 %s" % index_code)
    d = pd.DataFrame([x[:6] for x in rows],
                     columns=["trade_date", "open", "close", "high", "low", "volume"])
    d = d.drop_duplicates("trade_date").sort_values("trade_date").reset_index(drop=True)
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    for c in ("open", "close", "high", "low", "volume"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["amount"] = np.nan
    d["amplitude"] = np.nan
    d["pct_chg"] = d["close"].pct_change() * 100
    d["chg"] = d["close"].diff()
    d["turnover"] = np.nan
    return d[d["trade_date"] >= pd.Timestamp(str(beg))][COLS].reset_index(drop=True)


def download_index(beg="20100101"):
    IDX_DIR.mkdir(parents=True, exist_ok=True)
    out = []
    for code, (name, secid, fut) in INDEX.items():
        try:
            d = fetch_sse(code, beg)
        except Exception as e:
            print("  上交所拿 %s 失败（%s），改用腾讯备用源" % (name, type(e).__name__))
            d = fetch_tencent(code, beg)
        d.insert(1, "index_code", code)
        d.insert(2, "index_name", name)
        d.insert(3, "fut_product", fut)      # 对应的股指期货品种
        out.append(d)
        print("%-9s %s  %5d 行  %s ~ %s  收 %.2f" %
              (name, code, len(d), d["trade_date"].min().date(),
               d["trade_date"].max().date(), d["close"].iloc[-1]))
    a = pd.concat(out, ignore_index=True)
    p = IDX_DIR / "index_daily.parquet"
    # **必须合并，不能覆盖** —— daily_update 只抓最近 30 天，
    # 直接 to_parquet 会把 2010 年以来的历史全冲掉。
    if p.exists():
        a = pd.concat([pd.read_parquet(p), a], ignore_index=True)
    a = (a.drop_duplicates(["index_code", "trade_date"], keep="last")
          .sort_values(["index_code", "trade_date"]).reset_index(drop=True))
    # 合并后基于完整连续序列重算，避免增量窗口首日或切换备用源留下 NaN。
    a["pct_chg"] = a.groupby("index_code")["close"].pct_change() * 100
    a["chg"] = a.groupby("index_code")["close"].diff()
    tmp = p.with_name(p.name + ".tmp")
    a.to_parquet(tmp, index=False)
    os.replace(tmp, p)
    print("\n-> %s  %s 行 (%.1f MB)" % (p, format(len(a), ","), p.stat().st_size / 1e6))
    return a

## 18. 品种中文名

报告里要写「螺纹钢」不能写「rb」。大商所 xlsx 有「商品名称」列、广期所 csv 有「品种名称」、
上期所日更 dat 有 `PRODUCTNAME`，这三家直接从原始文件里抠，最可靠。
**只有郑商所的行情文件通篇没有中文**，那部分写死在 `CZCE_NAME` 里。

落到 `parsed/product_name.json`。出现新品种时这个 cell 重跑一次就行；
没有中文名的会退化成大写代码显示，不会报错（目前只有郑商所的 `PL` 是这样）。

In [ ]:
import zipfile

RAW = ROOT
NAME_JSON = PARSED / "product_name.json"

# 郑商所的行情文件里没有中文名，只能写死
CZCE_NAME = {
    "sr": "白糖", "cf": "棉花", "cy": "棉纱", "ta": "PTA", "ma": "甲醇", "fg": "玻璃",
    "sa": "纯碱", "ur": "尿素", "sf": "硅铁", "sm": "锰硅", "zc": "动力煤", "oi": "菜籽油",
    "rm": "菜籽粕", "rs": "油菜籽", "ap": "苹果", "cj": "红枣", "pf": "短纤", "pk": "花生",
    "sh": "烧碱", "px": "对二甲苯", "pr": "瓶片", "wh": "强麦", "pm": "普麦",
    "ri": "早籼稻", "lr": "晚籼稻", "jr": "粳稻",
    # 已退市的旧代码，历史数据里还在
    "er": "早籼稻(旧)", "me": "甲醇(旧)", "ro": "菜籽油(旧)", "tc": "动力煤(旧)",
    "ws": "强麦(旧)", "wt": "硬麦(旧)",
}
CFFEX_NAME = {
    "if": "沪深300", "ic": "中证500", "ih": "上证50", "im": "中证1000",
    "t": "10年国债", "tf": "5年国债", "ts": "2年国债", "tl": "30年国债",
}
# 带系列标记的月均价合约，split_contract 把标记拼进了 product
SERIES_NAME = {"ppf": "聚丙烯月均价", "lf": "聚乙烯月均价", "vf": "PVC月均价"}


def from_shfe():
    """日更 dat 的 PRODUCTNAME 字段。SHFE 这份是超集，INE 品种也在里面。"""
    h = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/121.0.0.0",
         "Referer": "https://www.shfe.com.cn/reports/tradedata/dailyandweeklydata/"}
    out = {}
    for day in pd.bdate_range(pd.Timestamp.today() - pd.Timedelta(days=10),
                              pd.Timestamp.today())[::-1]:
        u = "https://www.shfe.com.cn/data/tradedata/future/dailydata/kx%s.dat" % day.strftime("%Y%m%d")
        try:
            j = requests.get(u, headers=h, timeout=60).json().get("o_curinstrument") or []
        except Exception:
            continue
        for r in j:
            p = str(r.get("PRODUCTGROUPID", "")).strip().lower()
            n = str(r.get("PRODUCTNAME", "")).strip()
            if p and n and "_" not in p:
                out.setdefault(p, n)
        if out:
            break
    return out


def from_dce():
    """DCE 每个品种一个 xlsx，第一列就是「商品名称」。"""
    out = {}
    for z in sorted(RAW.glob("dce/history/DCE_futures_*.zip"), reverse=True)[:3]:
        with zipfile.ZipFile(z) as f:
            for m in f.namelist():
                if not m.lower().endswith((".xls", ".xlsx")):
                    continue
                try:
                    d = pd.read_excel(io.BytesIO(f.read(m)), nrows=1, engine="calamine")
                except Exception:
                    continue
                if "商品名称" not in d.columns or "合约名称" not in d.columns or not len(d):
                    continue
                import re
                c = str(d["合约名称"].iloc[0]).strip()
                p = re.match(r"^([A-Za-z]{1,3})", c)
                if not p:
                    continue
                key = p.group(1).lower()
                if m.lower().endswith("-f_ftr.xlsx") or "-F_" in m:
                    key += "f"
                out.setdefault(key, str(d["商品名称"].iloc[0]).strip())
    return out


def from_gfex():
    """GFEX csv 有「品种名称」列。"""
    out = {}
    for c in sorted(RAW.glob("gfex/history/GFEX_FUTURES_*.csv"), reverse=True)[:3]:
        try:
            d = pd.read_csv(c, skiprows=1, usecols=["品种名称", "合约代码"], dtype=str)
        except Exception:
            continue
        import re
        for name, con in d.drop_duplicates("品种名称").itertuples(index=False):
            m = re.match(r"^([A-Za-z]{1,3})", str(con).strip())
            if m:
                out.setdefault(m.group(1).lower(), str(name).strip())
    return out


def build_product_names():
    names = {}
    for src, fn in (("CZCE", lambda: CZCE_NAME), ("CFFEX", lambda: CFFEX_NAME),
                    ("系列", lambda: SERIES_NAME), ("GFEX", from_gfex),
                    ("DCE", from_dce), ("SHFE/INE", from_shfe)):
        got = fn()
        print("%-9s %2d 个" % (src, len(got)))
        names.update(got)
    NAME_JSON.parent.mkdir(parents=True, exist_ok=True)
    NAME_JSON.write_text(json.dumps(names, ensure_ascii=False, indent=1, sort_keys=True),
                   encoding="utf-8")
    print("\n-> %s  共 %d 个" % (NAME_JSON, len(names)))
    return names

## 19. 每日 HTML 报告

单文件 HTML，数据内联，**不引任何 CDN** —— 断网能开、发给别人也能开。
输出到 `报告/report_YYYYMMDD.html`。

内容：概览四个数字 + 主力合约表 + 次主力合约表（都只列持仓 > 10,000 手的）+ 股指期货基差表。
表头可点击排序。

### 三个口径上的决定

1. **涨跌用复权收益率**。换月当天新旧合约的收盘价直接相减是两个东西的价差，不是行情。
   正确做法是取新合约**自己**昨天的收盘价算当日涨跌，再把日收益连乘成复权序列，
   5 日 / 20 日从这条序列上取。实测 2025-06 以来 47 次换月里，**30 次**朴素算法会算错
   超过 0.5 个百分点 —— IC 2025-06-18 换月，朴素给 −3.22%，复权后其实只有 −0.22%。
2. **成交量和持仓量统一折算成单边**。大商所至今双边、郑商所 2020 年前双边，
   不折半的话大商所的持仓凭空大一倍，一张表里没法比大小。报告里**不显示口径标记**，
   直接给折算后的数 —— 这份是给非专业读者看的，多一列只会添乱。
3. **持仓变化和 5 日均量按合约自己算**，不按主力序列算。按序列算的话换月那天会把两个
   不同合约的量拼在一起，出现「成交 3,304 / 5 日均量 110,983」这种荒唐对比。

### 报告日期

默认取最后一个「六家齐全 + 结算价非空」的交易日。盘中用日更 dat 补进来的当天
只有 SHFE/INE、结算价还是空的，直接拿它出报告就是半截数据。

In [ ]:
import calendar, glob, html

OUTDIR = BASE / "报告"          # HTML 日报输出目录

EX_CN = {"SHFE": "上期所", "INE": "能源中心", "CZCE": "郑商所",
         "DCE": "大商所", "CFFEX": "中金所", "GFEX": "广期所"}
# 股指期货 → 标的指数
IDX_MAP = {"if": "000300", "ic": "000905", "im": "000852", "ih": "000016"}
IDX_ORDER = ["if", "ic", "im", "ih"]

RED, GREEN, GREY = "#c0392b", "#1a8a4a", "#8a8f98"

In [ ]:
# ------------------------------------------------------------------ 数据加载
def load_all():
    fut = pd.concat([pd.read_parquet(f) for f in
                     sorted(glob.glob(str(PARSED / "futures" / "futures_*.parquet")))[-3:]],
                    ignore_index=True)
    fut["trade_date"] = pd.to_datetime(fut["trade_date"])
    main = pd.read_parquet(DOM / "main.parquet")
    sub  = pd.read_parquet(DOM / "sub.parquet")
    idx  = pd.read_parquet(PARSED / "index" / "index_daily.parquet")
    names = json.loads((PARSED / "product_name.json").read_text(encoding="utf-8"))
    return fut, main, sub, idx, names


def as_date(x):
    """接受 None / 20260819 / "20260819" / "2026-08-19" / date，统一成 Timestamp。"""
    if x is None:
        return None
    if isinstance(x, (int, float)):
        x = "%d" % int(x)
    return pd.Timestamp(str(x))


def complete_dates(fut, min_rows=5, min_settled=0.9):
    """返回六所均达到动态行数门槛且关键字段覆盖完整的交易日。"""
    return complete_exchange_dates(fut, EX_CN, min_rows=min_rows,
                                   row_ratio=FUTURES_ROW_RATIO,
                                   min_coverage=min_settled)


def pick_date(fut, want=None):
    """取不晚于目标日的最后一个六所完整结算日。"""
    want = as_date(want)
    ok = complete_dates(fut)
    if want is not None:
        eligible = ok[ok <= want]
        if want in ok:
            return want
        print("  %s 数据不完整或非交易日，改用此前最后一个完整交易日" % want.date())
        ok = eligible
    if not len(ok):
        raise SystemExit("找不到完整的交易日")
    return ok.max()


# ------------------------------------------------------------ 口径 / 收益率
def to_single_side(df):
    """双边口径的成交量和持仓量折半，统一成单边。

    DCE 至今双边，郑商所 2020-01-01 之前双边。跨所放在一张表里比大小，
    不折半的话大商所的持仓凭空大一倍。

    **成交额跟成交量同口径**，所以也要折半。这一点是实测出来的：
    用 amount/(volume*settle) 反推合约乘数，27 个品种全部精确等于已知乘数
    （豆粕 10、铜 5、黄金 1000、沪深300 300……），说明两列是同一个口径。
    """
    d = df.copy()
    k = np.where(d["double_side"].fillna(False), 2.0, 1.0)
    for c in ("volume", "open_interest", "amount"):
        d[c] = d[c] / k
    return d


def adjusted_returns(tags, fut, windows=(1, 5, 20), since=None):
    """主力连续的复权收益率。

    since: 只算这个日期之后的序列。复权指数是连乘出来的，但 ret / ytd 都是
    两点之比，起点整体缩放不影响结果 —— 只要切点比「上一年最后一个交易日」
    和「往回 20 个交易日」都更早就行。

    换月当天不能拿新旧两个合约的收盘价相减 —— 那是两个东西的价差。
    正确做法是取**新合约自己**昨天的收盘价算当日涨跌，再把日收益连乘成复权序列，
    5 日 / 20 日收益从这条序列上取。这样换月的跳空自然被消掉。
    """
    if since is not None:
        since = pd.Timestamp(since)
        tags = tags[tags["trade_date"] >= since]
        fut = fut[fut["trade_date"] >= since]

    close = (fut[["trade_date", "contract", "close"]]
             .assign(close=lambda d: d["close"].where(d["close"] > 0))
             .drop_duplicates(["trade_date", "contract"])
             .rename(columns={"trade_date": "_pd", "close": "_prev"}))

    # 「当日合约在上一交易日的收盘价」用一次 merge 拿到，别在 Python 里逐行 .get()
    # —— 那是 20 万次 MultiIndex 查找，单这一行就 8.8s。
    t = tags.sort_values(["product", "trade_date"]).reset_index(drop=True)
    t["_pd"] = t.groupby("product", sort=False)["trade_date"].shift(1)
    t = t.merge(close, on=["_pd", "contract"], how="left")

    out = []
    for prod, g in t.groupby("product", sort=False):
        g = g.reset_index(drop=True)
        dates = list(g["trade_date"])
        cur   = g["close"].where(g["close"] > 0).astype(float).values

        prev = g["_prev"].astype(float).values.copy()
        prev[0] = np.nan                        # 第一天没有上一交易日
        r = cur / prev - 1
        idxs = np.concatenate([[1.0], np.nancumprod(1 + np.nan_to_num(r[1:]))])

        o = pd.DataFrame({"trade_date": dates, "product": prod})
        for w in windows:
            o["ret%d" % w] = pd.Series(idxs).pct_change(w).values

        # 年初至今：基准取**上一年最后一个交易日**的复权点位，
        # 不能取当年第一天 —— 那样 1 月 2 日的涨跌会被吞掉。
        # 当年才上市的品种没有上一年，就从它自己的第一个点算起。
        yrs = pd.Series(dates).dt.year.values
        starts = [0] + [i for i in range(1, len(yrs)) if yrs[i] != yrs[i - 1]]
        base = np.empty(len(o))
        for n, st in enumerate(starts):
            end = starts[n + 1] if n + 1 < len(starts) else len(o)
            base[st:end] = idxs[st - 1] if st > 0 else idxs[st]
        o["ytd"] = idxs / base - 1
        out.append(o)
    return pd.concat(out, ignore_index=True)


def contract_stats(fut1, date, lookback=180):
    """合约级的持仓变化和 5 日均量。

    必须按**合约自己**算，不能按品种的主力序列算 —— 换月那天主力序列会把两个
    不同合约的量拼在一起，出来的均量没有意义（实测次主力换月后出现
    「成交 3,304、5日均量 110,983」这种荒唐对比）。

    只有目标日那一行会被用到，所以先截掉 date 之前 lookback 天以外的数据再滚动。
    5 日窗口离 180 天远得很，含春节长假也够。
    """
    lo = pd.Timestamp(date) - pd.Timedelta(days=lookback)
    f = fut1[(fut1["trade_date"] >= lo) & (fut1["trade_date"] <= pd.Timestamp(date))]
    f = f.sort_values(["contract", "trade_date"])
    g = f.groupby("contract")
    return pd.DataFrame({
        "trade_date": f["trade_date"].values,
        "contract": f["contract"].values,
        "oi_d1": g["open_interest"].diff().values,
        "oi_d5": g["open_interest"].diff(5).values,
        "vol_ma5": g["volume"].transform(lambda x: x.rolling(5, min_periods=1).mean()).values,
    })


def build_table(tags, fut, date, min_oi=10000, only=None, windows=(1, 5, 20),
                fut1=None, stats=None):
    """only 给一组品种时，就不按 min_oi 过滤了，直接取这些品种 —— 次主力表用它
    跟主力表保持同一批品种，免得两张表对不上。

    fut1 / stats: 主力表和次主力表算出来的是同一份，由 prepare() 传进来复用。
    """
    tags = to_single_side(tags)
    fut1 = to_single_side(fut) if fut1 is None else fut1

    # ytd 的基准在上一年最后一个交易日，ret20 往回 20 个交易日 ——
    # 从上一年 10 月 1 日起算，两者都有充足余量。
    since = pd.Timestamp(year=pd.Timestamp(date).year - 1, month=10, day=1)
    rets = adjusted_returns(tags, fut1, windows, since=since)
    tags = tags.merge(rets, on=["trade_date", "product"], how="left")

    stats = contract_stats(fut1, date) if stats is None else stats
    tags = tags.merge(stats, on=["trade_date", "contract"], how="left")

    d = tags[tags["trade_date"] == date].copy()
    d = d[d["product"].isin(only)] if only is not None else d[d["open_interest"] > min_oi]
    return (d.sort_values("ret1", ascending=False, na_position="last")
             .reset_index(drop=True))

In [ ]:
# ---------------------------------------------------------------- 股指基差
def cffex_expiry(contract):
    """中金所股指期货：交割月第三个周五。"""
    m = re.search(r"(\d{4})\s*$", contract)
    y, mo = 2000 + int(m.group(1)[:2]), int(m.group(1)[2:])
    fri = [x for x in calendar.Calendar().itermonthdates(y, mo)
           if x.month == mo and x.weekday() == 4]
    return pd.Timestamp(fri[2])


def basis_frame(fut, idx, date, min_days=5):
    spot = (idx[idx["trade_date"] == date].set_index("index_code")["close"])
    f = fut[(fut["trade_date"] == date) & (fut["product"].isin(IDX_MAP))].copy()
    f = f[f["close"] > 0]
    if not len(f) or not len(spot):
        return pd.DataFrame()
    f["spot"] = f["product"].map(IDX_MAP).map(spot)
    f["expiry"] = f["contract"].map(cffex_expiry)
    f["days"] = (f["expiry"] - date).dt.days
    prev = (fut[fut["trade_date"] < date].sort_values("trade_date")
            .groupby("contract")["close"].last())
    f["fchg"] = f["close"] / f["contract"].map(prev).where(lambda x: x > 0) - 1
    f["basis"] = f["close"] - f["spot"]
    f["ann"] = f["basis"] / f["spot"] * 365 / f["days"].clip(lower=1) * 100
    return f[f["days"] >= min_days].sort_values(["product", "days"])




In [ ]:
# -------------------------------------------------------------------- HTML
def fmt_pct(v, tint=0.0):
    """tint 参数留着但不再用 —— 之前按 min/max 缩放的底色在邮件客户端里辨识度很差，
    现在统一只用红涨绿跌的文字颜色。"""
    if v is None or v != v:
        return '<td class="num dim">—</td>'
    cls = "up" if v > 0 else ("down" if v < 0 else "flat")
    return '<td class="num %s" data-v="%.6f">%+.2f%%</td>' % (cls, v, v * 100)


def fmt_int(v, signed=False):
    if v is None or v != v:
        return '<td class="num dim">—</td>'
    cls = ""
    if signed:
        cls = "up" if v > 0 else ("down" if v < 0 else "flat")
    s = format(v, "+,.0f") if signed else format(v, ",.0f")
    return '<td class="num %s" data-v="%.1f">%s</td>' % (cls, v, s)


def fmt_wan(v):
    """成交量 / 持仓统一用万手，保留一位小数 —— 原值七八位数太占宽度。"""
    if v is None or v != v:
        return '<td class="num dim">—</td>'
    return ('<td class="num" data-v="%.1f">%s<em>万</em></td>'
            % (v, format(v / 1e4, ",.1f")))


def fmt_price(v):
    """小数位按数值本身定：国债期货是 109.565，螺纹是 3017，别统一成两位。"""
    if v is None or v != v or v == 0:
        return '<td class="num dim">—</td>'
    s = format(round(v, 3), ",.3f").rstrip("0").rstrip(".")
    return '<td class="num" data-v="%.4f">%s</td>' % (v, s)


def basis_table_html(bf, idx, date, names, tid="t3"):
    """股指期货基差表：四个品种的全部在挂合约，按品种 + 到期先后排。"""
    if not len(bf):
        return "<p class='empty'>当日无股指期货数据</p>"
    idx_d = idx[idx["trade_date"] == date].set_index("index_code")
    heads = [("合约", "t"), ("期货收盘", "n"), ("指数点位", "n"), ("期货涨跌", "n"),
             ("指数涨跌", "n"), ("基差率", "n"), ("年化基差率", "n"),
             ("剩余天数", "n"), ("持仓", "n")]
    h = ['<div class="tw"><table id="%s"><thead><tr>' % tid]
    for i, (t, k) in enumerate(heads):
        h.append('<th class="%s" onclick="srt(&#39;%s&#39;,%d)">%s<i></i></th>' % (k, tid, i, t))
    h.append("</tr></thead><tbody>")

    order = {p: i for i, p in enumerate(IDX_ORDER)}
    bf = bf.assign(_o=bf["product"].map(order)).sort_values(["_o", "days"])
    for _, r in bf.iterrows():
        code = IDX_MAP[r["product"]]
        chg = float(idx_d.loc[code, "pct_chg"]) / 100 if code in idx_d.index else float("nan")
        h.append("<tr>")
        h.append('<td class="prod code">%s</td>' % html.escape(str(r["contract"])))
        h.append(fmt_price(r["close"]))
        h.append(fmt_price(r["spot"]))
        h.append(fmt_pct(r.get("fchg")))
        h.append(fmt_pct(chg))
        h.append(fmt_pct(r["basis"] / r["spot"]))
        a = r["ann"]
        h.append('<td class="num %s" data-v="%.4f">%+.2f%%</td>'
                 % ("up" if a > 0 else "down", a, a))
        h.append(fmt_int(r["days"]))
        h.append(fmt_wan(r["open_interest"]))
        h.append("</tr>")
    h.append("</tbody></table></div>")
    return "".join(h)


def table_html(d, names, tid):
    heads = [("品种", "t"), ("合约", "t"), ("收盘", "n"), ("1日", "n"), ("年初至今", "n"),
             ("成交量", "n"), ("5日均量", "n"), ("持仓", "n")]
    h = ['<div class="tw"><table id="%s"><thead><tr>' % tid]
    for i, (t, k) in enumerate(heads):
        h.append('<th class="%s" onclick="srt(\'%s\',%d)">%s<i></i></th>' % (k, tid, i, t))
    h.append("</tr></thead><tbody>")
    for _, r in d.iterrows():
        nm = names.get(r["product"], r["product"].upper())
        h.append("<tr>")
        h.append('<td class="prod">%s</td>' % html.escape(nm))
        h.append('<td class="code">%s</td>' % html.escape(str(r["contract"])))
        h.append(fmt_price(r["close"]))
        h.append(fmt_pct(r.get("ret1")))
        h.append(fmt_pct(r.get("ytd")))
        h.append(fmt_wan(r["volume"]))
        h.append(fmt_wan(r["vol_ma5"]))
        h.append(fmt_wan(r["open_interest"]))
        h.append("</tr>")
    h.append("</tbody></table></div>")
    return "".join(h)


CSS = """
*{box-sizing:border-box}
:root{
 --ink:#171a1f; --mut:#6b7280; --line:#e8eaee; --bg:#f4f6f9; --card:#fff;
 --up:#c8372d; --dn:#12855a; --accent:#2f5fd0; --amber:#b4761a;
}
body{margin:0;padding:0 0 56px;background:var(--bg);color:var(--ink);
 font:14px/1.5 "PingFang SC","Microsoft YaHei","Hiragino Sans GB",system-ui,sans-serif;
 -webkit-font-smoothing:antialiased}
.wrap{max-width:1180px;margin:0 auto;padding:0 18px}

/* 页头 */
.hero{background:linear-gradient(120deg,#1b2436 0%,#28405f 55%,#2f5fd0 140%);
 color:#fff;padding:26px 0 22px;margin-bottom:20px}
.hero .wrap{display:flex;align-items:flex-end;justify-content:space-between;gap:16px;flex-wrap:wrap}
.hero h1{font-size:25px;margin:0;letter-spacing:1px;font-weight:600}
.hero .d{font-size:13px;opacity:.78;margin-top:5px;line-height:1.7}
.hero .big{font-size:15px;opacity:.95;font-weight:600;letter-spacing:.5px}

/* 概览卡 */
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(178px,1fr));gap:11px;margin:0 0 6px}
.card{background:var(--card);border:1px solid var(--line);border-radius:9px;
 padding:11px 13px 12px;position:relative;overflow:hidden}
.card:before{content:"";position:absolute;left:0;top:0;bottom:0;width:3px;background:var(--accent)}
.card.c1:before{background:var(--up)} .card.c2:before{background:var(--accent)}
.card.c3:before{background:var(--amber)} .card.c4:before{background:var(--dn)}
.card .k{font-size:11.5px;color:var(--mut);letter-spacing:.3px}
.card .v{font-size:20px;font-weight:650;margin-top:2px;font-variant-numeric:tabular-nums}
.card .v small{font-size:12px;font-weight:500;color:var(--mut);margin-left:2px}

/* 标题 */
h2{font-size:17px;margin:30px 0 3px;display:flex;align-items:center;gap:8px;font-weight:600}
h2:before{content:"";width:4px;height:15px;border-radius:2px;background:var(--accent)}
h2.g:before{background:var(--amber)}
.note{color:var(--mut);font-size:12px;margin:5px 0 9px;line-height:1.65}

/* 表 */
.tw{overflow-x:auto;border:1px solid var(--line);border-radius:9px;background:var(--card);
 box-shadow:0 1px 2px rgba(16,24,40,.04)}
table{width:100%;border-collapse:separate;border-spacing:0;font-variant-numeric:tabular-nums}
th{position:sticky;top:0;z-index:2;background:#eef1f5;font-size:11.5px;font-weight:600;
 color:#3d434d;padding:7px 9px;white-space:nowrap;cursor:pointer;user-select:none;
 border-bottom:1px solid #dfe3e9;letter-spacing:.2px}
th.n{text-align:right}th.t{text-align:left}
th:hover{background:#e4e9f0;color:var(--accent)}
th i{display:inline-block;width:0;height:0;margin-left:3px;vertical-align:middle;opacity:.25;
 border-left:3.5px solid transparent;border-right:3.5px solid transparent;border-bottom:4.5px solid #333}
th.desc i{border-bottom:none;border-top:4.5px solid var(--accent);opacity:1}
th.asc i{border-bottom-color:var(--accent);opacity:1}
td{padding:4px 9px;border-bottom:1px solid #f2f3f5;font-size:13px;white-space:nowrap;line-height:1.45}
tbody tr:nth-child(even) td{background-color:#fbfcfd}
tbody tr:hover td{background-color:#f0f4fb}
tbody tr:last-child td{border-bottom:none}
.num{text-align:right}
.prod{font-weight:600;white-space:nowrap}
.code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:12px;color:#5b626d}
td em{font-style:normal;font-size:11px;color:var(--mut);margin-left:1px}
.up{color:var(--up)}.down{color:var(--dn)}.flat{color:var(--mut)}.dim{color:#c9cdd4}
.empty{color:#9ca3af;font-size:13px}
footer{margin-top:34px;padding-top:13px;border-top:1px solid var(--line);
 color:#9aa0a8;font-size:11.5px;line-height:1.8}
@media(max-width:720px){
 .wrap{padding:0 10px} .hero h1{font-size:21px}
 td,th{padding:4px 6px;font-size:12px}
}
"""

JS = """
function srt(id,i){
 var t=document.getElementById(id),b=t.tBodies[0],
     ths=t.tHead.rows[0].cells,th=ths[i],
     desc=!th.classList.contains('desc');
 for(var k=0;k<ths.length;k++){ths[k].classList.remove('asc','desc');}
 th.classList.add(desc?'desc':'asc');
 var rows=[].slice.call(b.rows);
 rows.sort(function(x,y){
   var a=x.cells[i],c=y.cells[i],
       av=a.dataset.v,cv=c.dataset.v;
   if(av===undefined&&cv===undefined)
     return a.textContent.localeCompare(c.textContent,'zh')*(desc?-1:1);
   if(av===undefined)return 1; if(cv===undefined)return -1;
   return (parseFloat(cv)-parseFloat(av))*(desc?1:-1);
 });
 rows.forEach(function(r){b.appendChild(r);});
}
"""


def prepare(date=None, min_oi=10000, loaded=None):
    """两种版式共用的那一份计算。完整版和邮件版都要它，算一次就够。"""
    fut, main, sub, idx, names = loaded or load_all()
    D = pick_date(fut, date)
    print("报告日期", D.date())

    fut1 = to_single_side(fut)              # 两张表共用，别算两遍
    stats = contract_stats(fut1, D)
    mt = build_table(main, fut, D, min_oi, fut1=fut1, stats=stats)
    st = build_table(sub,  fut, D, only=set(mt["product"]), fut1=fut1, stats=stats)
    miss = sorted(set(mt["product"]) - set(st["product"]))
    print("主力 %d 个（持仓 > %s，单边口径），次主力 %d 个%s"
          % (len(mt), format(min_oi, ","), len(st),
             ("，缺 " + " ".join(miss)) if miss else ""))
    return {"fut": fut, "idx": idx, "names": names, "D": D, "mt": mt, "st": st,
            "bf": basis_frame(fut, idx, D), "min_oi": min_oi}


def validate_report_ready(prep):
    """正式发信前的最后一道门：主/次主力和四个股指基差都必须齐全。"""
    D, idx, mt, st, bf = (prep[k] for k in ("D", "idx", "mt", "st", "bf"))
    mt_products = set(mt["product"].astype(str)) if "product" in mt else set()
    st_products = set(st["product"].astype(str)) if "product" in st else set()
    if not mt_products or st_products != mt_products:
        raise RuntimeError("主力/次主力表不完整：%d / %d" % (len(mt), len(st)))
    for label, table in (("主力", mt), ("次主力", st)):
        bad = [c for c in ("close", "volume", "open_interest")
               if c not in table or table[c].isna().any()]
        if bad:
            raise RuntimeError("%s表关键字段不完整：%s" % (label, ", ".join(bad)))
    idx_day = idx[idx["trade_date"].eq(D)].copy()
    usable = idx_day[idx_day["close"].notna() & idx_day["pct_chg"].notna()]
    missing_index = sorted(set(IDX_MAP.values()) - set(usable["index_code"].astype(str)))
    if missing_index:
        raise RuntimeError("报告日缺少股指现货数据：%s" % ", ".join(missing_index))
    basis_products = set(bf["product"].astype(str)) if "product" in bf else set()
    missing_basis = sorted(set(IDX_MAP) - basis_products)
    if missing_basis:
        raise RuntimeError("报告日缺少股指期货基差：%s" % ", ".join(x.upper() for x in missing_basis))
    return True

In [ ]:
def build_report(date=None, min_oi=10000, out=None, prep=None):
    prep = prep or prepare(date, min_oi)
    fut, idx, names = prep["fut"], prep["idx"], prep["names"]
    D, mt, st, min_oi = prep["D"], prep["mt"], prep["st"], prep["min_oi"]

    # 概览
    day = to_single_side(fut[fut["trade_date"] == D])
    up = int((mt["ret1"] > 0).sum()); dn = int((mt["ret1"] < 0).sum())
    cards = [("上涨 / 下跌品种", '<span class="up">%d</span> / <span class="down">%d</span>' % (up, dn)),
             ("全市场成交量", "%.0f<small>万手</small>" % (day["volume"].sum() / 1e4)),
             ("全市场持仓量", "%.0f<small>万手</small>" % (day["open_interest"].sum() / 1e4)),
             ("全市场成交额", "%.0f<small>亿元</small>" % (day["amount"].sum() / 1e8))]

    # 股指基差
    bf = prep["bf"]
    bsum = ""
    if len(bf):
        near = bf.sort_values(["product", "days"]).groupby("product").first()
        bsum = "；".join("%s %+.1f%%" % (p.upper(), near.loc[p, "ann"])
                         for p in IDX_ORDER if p in near.index)

    h = ['<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">',
         "<title>期货市场日报 %s</title>" % D.strftime("%Y-%m-%d"),
         "<style>%s</style>" % CSS,
         '<div class="hero"><div class="wrap"><div>',
         "<h1>期货市场日报</h1>",
         '<div class="d">六家交易所 &nbsp;·&nbsp; 成交量与持仓量为单边口径 &nbsp;·&nbsp; '
         '涨跌为主力连续复权收益率</div></div>',
         '<div class="big">%s</div>' % D.strftime("%Y年%m月%d日"),
         '</div></div><div class="wrap">',
         '<div class="cards">']
    for n, (k, v) in enumerate(cards, 1):
        h.append('<div class="card c%d"><div class="k">%s</div><div class="v">%s</div></div>'
                 % (n, k, v))
    h.append("</div>")

    h.append('<h2 class="g">股指期货基差</h2>')
    h.append('<div class="note">负值即贴水，做多股指期货相当于每年多赚这个百分比。'
             '剩余不足 5 天的合约不列。</div>')
    h.append(basis_table_html(bf, idx, D, names))

    h.append("<h2>主力合约</h2>")
    h.append('<div class="note">持仓量 &gt; %s 手，共 %d 个品种，按当日涨跌从高到低排。'
             '点表头可换其它排序。</div>'
             % (format(min_oi, ","), len(mt)))
    h.append(table_html(mt, names, "t1"))

    h.append("<h2>次主力合约</h2>")
    h.append('<div class="note">品种跟上表一致，同样按当日涨跌从高到低排，共 %d 个。</div>'
             % len(st))
    h.append(table_html(st, names, "t2"))

    h.append("<footer>数据来源：上期所 / 能源中心 / 郑商所 / 大商所 / 中金所 / 广期所官方历史行情，"
             "指数优先采用上交所官方行情、腾讯备用。主力合约的换月规则：某合约收盘持仓量超过当前主力 1.1 倍时，"
             "下一交易日起切换，且只能切到交割月不早于当前主力的合约。"
             "本报告仅供参考，不构成投资建议。<br>生成时间 %s</footer>"
             % dt.datetime.now().strftime("%Y-%m-%d %H:%M"))
    h.append("</div><script>%s</script>" % JS)

    OUTDIR.mkdir(parents=True, exist_ok=True)
    out = Path(out) if out else OUTDIR / ("report_%s.html" % D.strftime("%Y%m%d"))
    out.write_text("\n".join(h), encoding="utf-8")
    print("-> %s  (%.0f KB)" % (out, out.stat().st_size / 1024))
    return out

## 20. 完整流程

### 首次全量

```python
download_all()        # 1. 下载约 1.3 GB（往年 skip，当年强制刷新）
build_all()           # 2. 解析 → parquet，约 25 分钟
fill_shfe_tail()      # 3. 补 SHFE/INE 年度包的尾巴
audit()               # 4. 覆盖体检
recon(2024)           # 5. 跟米筐 daily_contracts 对账
build_dominant()      # 6. 主力 / 次主力落文件
download_index()      # 7. 股指标的指数（上交所官方主源，腾讯备用）
build_product_names() # 8. 品种中文名
build_report()        # 9. 出 HTML 日报
```

### 之后每个交易日

```python
daily_update()        # 下载 → 解析 → 补尾巴 → 重算主力 → 出 HTML 日报
```

只要行情、暂时不重算主力：`daily_update(rebuild_dominant=False)`。

### 还没做的

- **跟米筐对账**：`rqdatac.all_instruments(type='Future')` + `get_price(...)`，
  逐日比 close / settle / open_interest。对之前记得先按 `double_side` 归一口径。
- `oi_chg` 在 SHFE / INE 是 100% 空的 —— 它们的报表里没有「持仓量变化」这一列，
  只有日更 dat 有。往年要补只能按合约对 `open_interest` 做 diff，首日会是 NaN。

In [ ]:
if __name__ == "__main__":
    daily_update()